In [ ]:
# ============================================================
# DPPU-VRU v21 -- TEACH THE PROCESS NOT THE ANSWER
# Dylan Michael Scott -- Horizon Tech
#
# Instead of: 73+18=91
# We give it:  73+18=ones:11,carry:1,tens:9,ans:91
#
# The model follows the actual computation steps.
# Ones column first. Carry. Tens column. Then answer.
# This mirrors how arithmetic is actually performed --
# not as a lookup, but as a sequential process.
#
# DPPU vs Vanilla side by side. Same batches. Honest test.
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import torch, torch.nn as nn, torch.nn.functional as F
import math, random, os, json, sys
from datetime import datetime

DRIVE_DIR = '/content/drive/MyDrive/dppu_vru'
LOG_PATH  = os.path.join(DRIVE_DIR, 'v21_log.txt')
os.makedirs(DRIVE_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

PI_CL  = math.pi
PHI_CL = 4.0 / math.pi
D_CAP  = 4
EPS    = 1e-7

def pi_dyn(d):  return 4.0 - (4.0 - PI_CL) * torch.exp(-d)
def phi_dyn(d): return PHI_CL * torch.exp(-d) + (1.0 - torch.exp(-d))
def omega(d):   return (pi_dyn(d) * phi_dyn(d)) / (1.0 + d + EPS)

def compute_delta(h):
    return torch.tanh(torch.abs(h) / (h.std(dim=-1, keepdim=True) + EPS))

CFG = dict(
    hidden       = 260,
    lr           = 1e-3,
    batch        = 32,
    max_steps    = 20_000,
    log_every    = 200,
    probe_every  = 2_000,
    tf_start     = 0.9,
    tf_min       = 0.05,
    tf_decay     = 0.9995,
)

# ============================================================
# TOKENIZER -- extended for step tokens
# ============================================================
class MathTokenizer:
    SPECIAL = ['<pad>', '<bos>', '<eos>', '<unk>']

    def __init__(self):
        self.vocab = {s: i for i, s in enumerate(self.SPECIAL)}
        # digits, operators, separators
        for c in '0123456789+-*/=:,abcdefghijklmnopqrstuvwxyz_ ':
            if c not in self.vocab:
                self.vocab[c] = len(self.vocab)
        self.inv = {v: k for k, v in self.vocab.items()}
        self.pad_id = self.vocab['<pad>']
        self.bos_id = self.vocab['<bos>']
        self.eos_id = self.vocab['<eos>']
        self.eq_id  = self.vocab['=']

    @property
    def vocab_size(self): return len(self.vocab)

    def encode(self, text, add_bos=True, add_eos=True):
        return (([self.bos_id] if add_bos else []) +
                [self.vocab.get(c, self.vocab['<unk>']) for c in text] +
                ([self.eos_id] if add_eos else []))

    def decode(self, ids):
        return ''.join(self.inv.get(i,'?') for i in ids
                       if self.inv.get(i,'?') not in self.SPECIAL)

# ============================================================
# DATA -- step-by-step format
#
# 73+18= -> ones:11,carry:1,tens:9,ans:91
#
# The answer is still at the end (ans:XX)
# but the model has to work through the process to get there
# ============================================================
def make_step_example(a, b):
    ones      = (a % 10) + (b % 10)
    carry     = ones // 10
    tens_sum  = (a // 10) + (b // 10) + carry
    answer    = a + b
    # format: question=ones:N,carry:N,tens:N,ans:N
    prompt  = f"{a}+{b}="
    process = f"ones:{ones},carry:{carry},tens:{tens_sum},ans:{answer}"
    return prompt + process

def gen_add2():
    a, b = random.randint(10,99), random.randint(10,99)
    return make_step_example(a, b)

def make_batch(bsz, tok):
    seqs, starts = [], []
    for _ in range(bsz):
        raw   = gen_add2()
        ids   = tok.encode(raw)
        # answer starts after '=' -- model predicts the process
        start = next((j+1 for j,t in enumerate(ids) if t==tok.eq_id), 1)
        seqs.append(ids); starts.append(start)
    ml  = max(len(s) for s in seqs)
    pad = [s + [tok.pad_id]*(ml-len(s)) for s in seqs]
    return torch.tensor(pad, dtype=torch.long, device=device), starts

# ============================================================
# DPPU MODEL
# ============================================================
class DPPUCell(nn.Module):
    def __init__(self, in_dim, hid):
        super().__init__()
        self.n  = D_CAP + 1
        self.ds = hid // self.n
        self.Wx = nn.Linear(in_dim, hid)
        self.Wh = nn.ModuleList([nn.Linear(hid, self.ds, bias=False) for _ in range(self.n)])
        self.Wc = nn.Linear(hid, self.n, bias=False)
        self.Wo = nn.Linear(hid, hid, bias=False)
        self.anchor = nn.Parameter(torch.tensor(PHI_CL))

    def forward(self, x, h, C):
        xp, parts = self.Wx(x), []
        for i in range(self.n):
            s, e   = i*self.ds, (i+1)*self.ds
            di     = compute_delta(h[:, s:e])
            h_rec  = (phi_dyn(di) / PHI_CL) * self.Wh[i](h)
            x_rec  = (pi_dyn(di)  / PI_CL)  * xp[:, s:e]
            gate   = torch.sigmoid(omega(di))
            anchor = C[:, i:i+1] * self.anchor
            parts.append(torch.tanh(h_rec + x_rec + anchor) * gate)
        h2 = self.Wo(torch.cat(parts, dim=-1))
        C2 = torch.clamp(torch.tanh(0.1*C + 0.01*self.Wc(h2)), -0.5, 0.5)
        return h2, C2


class DPPUModel(nn.Module):
    def __init__(self, vsz, hid=260):
        super().__init__()
        self.hid  = hid
        self.n    = D_CAP + 1
        self.emb  = nn.Embedding(vsz, hid)
        self.cell = DPPUCell(hid, hid)
        self.out  = nn.Linear(hid, vsz)

    def zeros(self, B):
        h = torch.zeros(B, self.hid, device=device)
        C = torch.full((B, self.n), PHI_CL, device=device)
        return h, C

    def forward(self, ids, tf=1.0):
        B, T = ids.shape
        h, C = self.zeros(B)
        outs  = []
        for t in range(T-1):
            tok  = ids[:,t] if (t==0 or random.random()<tf) else outs[-1].argmax(-1)
            x    = self.emb(tok)
            h, C = self.cell(x, h, C)
            outs.append(self.out(h))
        return torch.stack(outs, dim=1)

# ============================================================
# VANILLA MODEL
# ============================================================
class VanillaModel(nn.Module):
    def __init__(self, vsz, hid=260):
        super().__init__()
        self.hid  = hid
        self.emb  = nn.Embedding(vsz, hid)
        self.cell = nn.RNNCell(hid, hid)
        self.out  = nn.Linear(hid, vsz)

    def forward(self, ids, tf=1.0):
        B, T = ids.shape
        h    = torch.zeros(B, self.hid, device=device)
        outs = []
        for t in range(T-1):
            tok  = ids[:,t] if (t==0 or random.random()<tf) else outs[-1].argmax(-1)
            x    = self.emb(tok)
            h    = self.cell(x, h)
            outs.append(self.out(h))
        return torch.stack(outs, dim=1)

# ============================================================
# LOSS
# ============================================================
def loss_and_acc(logits, ids, starts, pad_id):
    B = logits.size(0)
    L = torch.tensor(0.0, device=logits.device)
    ok = tot = 0
    for b in range(B):
        for t in range(starts[b]-1, ids.size(1)-1):
            tgt = ids[b,t+1].item()
            if tgt == pad_id: break
            L += F.cross_entropy(logits[b,t].unsqueeze(0),
                                 torch.tensor([tgt], device=logits.device))
            if logits[b,t].argmax(-1).item() == tgt: ok += 1
            tot += 1
    return (L/tot, ok/tot) if tot>0 else (L, 0.0)

# ============================================================
# GENERATION
# ============================================================
@torch.no_grad()
def gen_dppu(model, tok, prompt_ids, max_len=30):
    inp     = torch.tensor([prompt_ids], dtype=torch.long, device=device)
    h, C    = model.zeros(1)
    for t in range(inp.size(1)-1):
        x    = model.emb(inp[:,t])
        h, C = model.cell(x, h, C)
    cur, out = inp[:,-1], []
    for _ in range(max_len):
        x    = model.emb(cur)
        h, C = model.cell(x, h, C)
        lg   = model.out(h)
        nxt  = lg.argmax(-1)
        char = tok.inv.get(nxt.item(), '?')
        if char in ('<eos>','<pad>'): break
        out.append(char); cur = nxt
    return ''.join(out)

@torch.no_grad()
def gen_vanilla(model, tok, prompt_ids, max_len=30):
    inp = torch.tensor([prompt_ids], dtype=torch.long, device=device)
    h   = torch.zeros(1, model.hid, device=device)
    for t in range(inp.size(1)-1):
        x = model.emb(inp[:,t])
        h = model.cell(x, h)
    cur, out = inp[:,-1], []
    for _ in range(max_len):
        x  = model.emb(cur)
        h  = model.cell(x, h)
        lg = model.out(h)
        nxt  = lg.argmax(-1)
        char = tok.inv.get(nxt.item(), '?')
        if char in ('<eos>','<pad>'): break
        out.append(char); cur = nxt
    return ''.join(out)

def extract_answer(output):
    # pull the number after 'ans:' if present
    if 'ans:' in output:
        try: return output.split('ans:')[-1].split(',')[0].strip()
        except: pass
    return output

# ============================================================
# PROBE
# ============================================================
PROBE_EX = [
    (12, 34, 46),
    (55, 27, 82),
    (73, 18, 91),
    (99, 11, 110),
    (64, 36, 100),
]

@torch.no_grad()
def run_probe(dppu, vanilla, tok, step):
    dppu.eval(); vanilla.eval()
    d_ok = v_ok = 0
    lines = [f"\n  PROBE step={step}  anchor={dppu.cell.anchor.item():.6f}"]
    lines.append(f"  {'PROBLEM':<12} {'TARGET':<6} {'DPPU OUTPUT':<28} {'VANILLA OUTPUT':<28} {'ANS'}")
    lines.append(f"  {'-'*85}")
    for a, b, ans in PROBE_EX:
        prompt  = f"{a}+{b}="
        target  = make_step_example(a, b).split('=')[1]
        ids     = tok.encode(prompt, add_bos=True, add_eos=False)
        dp_out  = gen_dppu(dppu, tok, ids)
        vp_out  = gen_vanilla(vanilla, tok, ids)
        dp_ans  = extract_answer(dp_out)
        vp_ans  = extract_answer(vp_out)
        dh = '✓' if dp_ans == str(ans) else '✗'
        vh = '✓' if vp_ans == str(ans) else '✗'
        if dp_ans == str(ans): d_ok += 1
        if vp_ans == str(ans): v_ok += 1
        lines.append(f"  {prompt:<12} {str(ans):<6} {dh} {dp_out:<26} {vh} {vp_out:<26}")
    lines.append(f"\n  DPPU: {d_ok}/5 correct answers    VANILLA: {v_ok}/5 correct answers\n")
    msg = '\n'.join(lines)
    print(msg)
    with open(LOG_PATH,'a') as f: f.write(msg+'\n')
    sys.stdout.flush()
    dppu.train(); vanilla.train()

# ============================================================
# MAIN
# ============================================================
tok     = MathTokenizer()
dppu    = DPPUModel(tok.vocab_size, CFG['hidden']).to(device)
vanilla = VanillaModel(tok.vocab_size, CFG['hidden']).to(device)
d_opt   = torch.optim.Adam(dppu.parameters(),    lr=CFG['lr'])
v_opt   = torch.optim.Adam(vanilla.parameters(), lr=CFG['lr'])

d_params = sum(p.numel() for p in dppu.parameters())
v_params = sum(p.numel() for p in vanilla.parameters())

# print a sample to show the format
sample = gen_add2()
hdr = (f"\n{'='*62}\n"
       f"  DPPU-VRU v21  |  TEACH THE PROCESS\n"
       f"  device={device}\n"
       f"  DPPU params={d_params:,}  |  Vanilla params={v_params:,}\n"
       f"\n  Example training sequence:\n"
       f"  INPUT:  {sample.split('=')[0]}=\n"
       f"  TARGET: {sample.split('=')[1]}\n"
       f"\n  Model must predict: ones -> carry -> tens -> answer\n"
       f"  Not just the answer. The whole process.\n"
       f"{'='*62}\n")
print(hdr)
with open(LOG_PATH,'a') as f: f.write(hdr)

tf = CFG['tf_start']
dl = da = vl = va = rc = 0.0

for step in range(1, CFG['max_steps']+1):
    tf = max(CFG['tf_min'], tf * CFG['tf_decay'])
    ids, starts = make_batch(CFG['batch'], tok)

    # DPPU
    dppu.train()
    d_opt.zero_grad()
    d_logits = dppu(ids, tf=tf)
    d_loss, d_acc = loss_and_acc(d_logits, ids, starts, tok.pad_id)
    d_loss.backward()
    torch.nn.utils.clip_grad_norm_(dppu.parameters(), 5.0)
    d_opt.step()

    # Vanilla
    vanilla.train()
    v_opt.zero_grad()
    v_logits = vanilla(ids, tf=tf)
    v_loss, v_acc = loss_and_acc(v_logits, ids, starts, tok.pad_id)
    v_loss.backward()
    torch.nn.utils.clip_grad_norm_(vanilla.parameters(), 5.0)
    v_opt.step()

    dl += d_loss.item(); da += d_acc
    vl += v_loss.item(); va += v_acc
    rc += 1

    if step % CFG['log_every'] == 0:
        msg = (f"step={step:6d} | "
               f"DPPU  loss={dl/rc:.4f} acc={da/rc*100:.1f}%  anchor={dppu.cell.anchor.item():.4f} | "
               f"VANILLA  loss={vl/rc:.4f} acc={va/rc*100:.1f}% | "
               f"tf={tf:.3f}")
        print(msg)
        with open(LOG_PATH,'a') as f: f.write(msg+'\n')
        sys.stdout.flush()
        dl = da = vl = va = rc = 0.0

    if step % CFG['probe_every'] == 0:
        run_probe(dppu, vanilla, tok, step)

print(f"\nDONE -- step={step}")
print(f"Final anchor: {dppu.cell.anchor.item():.6f}  (started at {PHI_CL:.6f})")

Mounted at /content/drive

  DPPU-VRU v21  |  TEACH THE PROCESS
  device=cuda
  DPPU params=229,890  |  Vanilla params=161,249

  Example training sequence:
  INPUT:  75+47=
  TARGET: ones:12,carry:1,tens:12,ans:122

  Model must predict: ones -> carry -> tens -> answer
  Not just the answer. The whole process.

step=   200 | DPPU  loss=0.5953 acc=82.6%  anchor=1.3088 | VANILLA  loss=0.6046 acc=82.7% | tf=0.814
step=   400 | DPPU  loss=0.3980 acc=86.9%  anchor=1.3312 | VANILLA  loss=0.3952 acc=86.4% | tf=0.737
step=   600 | DPPU  loss=0.4168 acc=86.2%  anchor=1.3399 | VANILLA  loss=0.3838 acc=86.9% | tf=0.667
step=   800 | DPPU  loss=0.4088 acc=86.1%  anchor=1.3402 | VANILLA  loss=0.4049 acc=86.2% | tf=0.603
step=  1000 | DPPU  loss=0.4493 acc=85.2%  anchor=1.3386 | VANILLA  loss=0.4008 acc=86.3% | tf=0.546


In [ ]:
# ============================================================
# DPPU-VRU v22 -- DUAL HEMISPHERE + AGREEMENT GATE
# Dylan Michael Scott -- Horizon Tech
#
# Architecture inspired by bilateral cognition:
#   - Two processing streams (left/right hemisphere)
#   - Agreement gate: output only when both streams align
#   - Neutral state: hold when streams disagree
#   - Entropy tracked for shape, not just magnitude
#
# Format: step-by-step arithmetic (from v21)
#   73+18=ones:11,carry:1,tens:9,ans:91
#
# Single cell comparison: DPPU-v22 vs Vanilla
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import torch, torch.nn as nn, torch.nn.functional as F
import math, random, os, sys
from datetime import datetime

DRIVE_DIR = '/content/drive/MyDrive/dppu_vru'
LOG_PATH  = os.path.join(DRIVE_DIR, 'v22_log.txt')
os.makedirs(DRIVE_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

PI_CL  = math.pi
PHI_CL = 4.0 / math.pi
D_CAP  = 4
EPS    = 1e-7

def pi_dyn(d):  return 4.0 - (4.0 - PI_CL) * torch.exp(-d)
def phi_dyn(d): return PHI_CL * torch.exp(-d) + (1.0 - torch.exp(-d))
def omega(d):   return (pi_dyn(d) * phi_dyn(d)) / (1.0 + d + EPS)

def compute_delta(h):
    return torch.tanh(torch.abs(h) / (h.std(dim=-1, keepdim=True) + EPS))

CFG = dict(
    hidden       = 260,
    lr           = 1e-3,
    batch        = 32,
    max_steps    = 20_000,
    log_every    = 200,
    probe_every  = 2_000,
    tf_start     = 0.9,
    tf_min       = 0.05,
    tf_decay     = 0.9995,
)

# ============================================================
# TOKENIZER
# ============================================================
class MathTokenizer:
    SPECIAL = ['<pad>', '<bos>', '<eos>', '<unk>']

    def __init__(self):
        self.vocab = {s: i for i, s in enumerate(self.SPECIAL)}
        for c in '0123456789+-*/=:,abcdefghijklmnopqrstuvwxyz_ ':
            if c not in self.vocab:
                self.vocab[c] = len(self.vocab)
        self.inv = {v: k for k, v in self.vocab.items()}
        self.pad_id = self.vocab['<pad>']
        self.bos_id = self.vocab['<bos>']
        self.eos_id = self.vocab['<eos>']
        self.eq_id  = self.vocab['=']

    @property
    def vocab_size(self): return len(self.vocab)

    def encode(self, text, add_bos=True, add_eos=True):
        return (([self.bos_id] if add_bos else []) +
                [self.vocab.get(c, self.vocab['<unk>']) for c in text] +
                ([self.eos_id] if add_eos else []))

    def decode(self, ids):
        return ''.join(self.inv.get(i,'?') for i in ids
                       if self.inv.get(i,'?') not in self.SPECIAL)

# ============================================================
# DATA -- step by step format (v21 proven)
# ============================================================
def make_step_example(a, b):
    ones     = (a % 10) + (b % 10)
    carry    = ones // 10
    tens_sum = (a // 10) + (b // 10) + carry
    return f"{a}+{b}=ones:{ones},carry:{carry},tens:{tens_sum},ans:{a+b}"

def gen_add2():
    a, b = random.randint(10,99), random.randint(10,99)
    return make_step_example(a, b)

def make_batch(bsz, tok):
    seqs, starts = [], []
    for _ in range(bsz):
        raw   = gen_add2()
        ids   = tok.encode(raw)
        start = next((j+1 for j,t in enumerate(ids) if t==tok.eq_id), 1)
        seqs.append(ids); starts.append(start)
    ml  = max(len(s) for s in seqs)
    pad = [s + [tok.pad_id]*(ml-len(s)) for s in seqs]
    return torch.tensor(pad, dtype=torch.long, device=device), starts

# ============================================================
# DPPU HEMISPHERE CELL
# One hemisphere -- half the hidden dim
# ============================================================
class DPPUHemisphere(nn.Module):
    def __init__(self, in_dim, hid):
        super().__init__()
        self.n  = D_CAP + 1
        self.ds = hid // self.n
        self.Wx = nn.Linear(in_dim, hid)
        self.Wh = nn.ModuleList([nn.Linear(hid, self.ds, bias=False) for _ in range(self.n)])
        self.Wc = nn.Linear(hid, self.n, bias=False)
        self.Wo = nn.Linear(hid, hid, bias=False)
        self.anchor = nn.Parameter(torch.tensor(PHI_CL))

    def forward(self, x, h, C):
        xp, parts = self.Wx(x), []
        for i in range(self.n):
            s, e   = i*self.ds, (i+1)*self.ds
            di     = compute_delta(h[:, s:e])
            h_rec  = (phi_dyn(di) / PHI_CL) * self.Wh[i](h)
            x_rec  = (pi_dyn(di)  / PI_CL)  * xp[:, s:e]
            gate   = torch.sigmoid(omega(di))
            anchor = C[:, i:i+1] * self.anchor
            parts.append(torch.tanh(h_rec + x_rec + anchor) * gate)
        h2 = self.Wo(torch.cat(parts, dim=-1))
        C2 = torch.clamp(torch.tanh(0.1*C + 0.01*self.Wc(h2)), -0.5, 0.5)
        return h2, C2


# ============================================================
# DUAL HEMISPHERE MODEL WITH AGREEMENT GATE
# ============================================================
class DualHemisphereModel(nn.Module):
    def __init__(self, vsz, hid=260):
        super().__init__()
        self.hid  = hid
        self.n    = D_CAP + 1
        self.emb  = nn.Embedding(vsz, hid)

        # two independent hemisphere cells
        self.left  = DPPUHemisphere(hid, hid)
        self.right = DPPUHemisphere(hid, hid)

        # agreement gate -- learns when to trust the output
        # fires when left and right are aligned
        self.agree = nn.Linear(hid * 2, hid)

        # focus layer -- the frontal arbitration
        # takes combined hemispheres + agreement signal
        self.focus = nn.Linear(hid * 2 + hid, hid)

        self.out   = nn.Linear(hid, vsz)

    def zeros(self, B):
        h = torch.zeros(B, self.hid, device=device)
        C = torch.full((B, self.n), PHI_CL, device=device)
        return h, C

    def forward(self, ids, tf=1.0):
        B, T = ids.shape
        lh, lC = self.zeros(B)
        rh, rC = self.zeros(B)
        outs = []

        for t in range(T-1):
            tok = ids[:,t] if (t==0 or random.random()<tf) else outs[-1].argmax(-1)
            x   = self.emb(tok)

            # left hemisphere processes normally
            lh, lC = self.left(x, lh, lC)

            # right hemisphere gets same input but independent state
            rh, rC = self.right(x, rh, rC)

            # agreement gate -- how aligned are the two streams?
            # dot product similarity as the agreement signal
            combined  = torch.cat([lh, rh], dim=-1)
            agreement = torch.sigmoid(self.agree(combined))  # (B, hid)

            # cosine similarity between hemispheres -- the focus signal
            cos_sim = F.cosine_similarity(lh, rh, dim=-1, eps=EPS).unsqueeze(-1)  # (B, 1)

            # focus: agreement-weighted blend of both hemispheres
            # when cos_sim high (agreement) -> strong clear signal
            # when cos_sim low (disagreement) -> muted, hold back
            focused = self.focus(torch.cat([combined, agreement], dim=-1))
            focused = focused * torch.sigmoid(cos_sim * 4.0)  # gate sharpens around 0

            outs.append(self.out(focused))

        return torch.stack(outs, dim=1)


# ============================================================
# VANILLA BASELINE
# ============================================================
class VanillaModel(nn.Module):
    def __init__(self, vsz, hid=260):
        super().__init__()
        self.hid  = hid
        self.emb  = nn.Embedding(vsz, hid)
        self.cell = nn.RNNCell(hid, hid)
        self.out  = nn.Linear(hid, vsz)

    def forward(self, ids, tf=1.0):
        B, T = ids.shape
        h    = torch.zeros(B, self.hid, device=device)
        outs = []
        for t in range(T-1):
            tok  = ids[:,t] if (t==0 or random.random()<tf) else outs[-1].argmax(-1)
            x    = self.emb(tok)
            h    = self.cell(x, h)
            outs.append(self.out(h))
        return torch.stack(outs, dim=1)

# ============================================================
# LOSS
# ============================================================
def loss_and_acc(logits, ids, starts, pad_id):
    B = logits.size(0)
    L = torch.tensor(0.0, device=logits.device)
    ok = tot = 0
    for b in range(B):
        for t in range(starts[b]-1, ids.size(1)-1):
            tgt = ids[b,t+1].item()
            if tgt == pad_id: break
            L += F.cross_entropy(logits[b,t].unsqueeze(0),
                                 torch.tensor([tgt], device=logits.device))
            if logits[b,t].argmax(-1).item() == tgt: ok += 1
            tot += 1
    return (L/tot, ok/tot) if tot>0 else (L, 0.0)

def mean_entropy(logits):
    p = torch.softmax(logits, dim=-1)
    return -(p * torch.log(p + EPS)).sum(-1).mean().item()

# ============================================================
# GENERATION
# ============================================================
@torch.no_grad()
def gen_dual(model, tok, prompt_ids, max_len=30):
    inp     = torch.tensor([prompt_ids], dtype=torch.long, device=device)
    lh, lC  = model.zeros(1)
    rh, rC  = model.zeros(1)
    for t in range(inp.size(1)-1):
        x       = model.emb(inp[:,t])
        lh, lC  = model.left(x, lh, lC)
        rh, rC  = model.right(x, rh, rC)
    cur, out = inp[:,-1], []
    for _ in range(max_len):
        x        = model.emb(cur)
        lh, lC   = model.left(x, lh, lC)
        rh, rC   = model.right(x, rh, rC)
        combined  = torch.cat([lh, rh], dim=-1)
        agreement = torch.sigmoid(model.agree(combined))
        cos_sim   = F.cosine_similarity(lh, rh, dim=-1, eps=EPS).unsqueeze(-1)
        focused   = model.focus(torch.cat([combined, agreement], dim=-1))
        focused   = focused * torch.sigmoid(cos_sim * 4.0)
        lg        = model.out(focused)
        nxt       = lg.argmax(-1)
        char      = tok.inv.get(nxt.item(), '?')
        if char in ('<eos>','<pad>'): break
        out.append(char); cur = nxt
    return ''.join(out)

@torch.no_grad()
def gen_vanilla(model, tok, prompt_ids, max_len=30):
    inp = torch.tensor([prompt_ids], dtype=torch.long, device=device)
    h   = torch.zeros(1, model.hid, device=device)
    for t in range(inp.size(1)-1):
        x = model.emb(inp[:,t])
        h = model.cell(x, h)
    cur, out = inp[:,-1], []
    for _ in range(max_len):
        x    = model.emb(cur)
        h    = model.cell(x, h)
        lg   = model.out(h)
        nxt  = lg.argmax(-1)
        char = tok.inv.get(nxt.item(), '?')
        if char in ('<eos>','<pad>'): break
        out.append(char); cur = nxt
    return ''.join(out)

def extract_answer(output):
    if 'ans:' in output:
        try: return output.split('ans:')[-1].split(',')[0].strip()
        except: pass
    return output

# ============================================================
# PROBE
# ============================================================
PROBE_EX = [
    (12, 34, 46),
    (55, 27, 82),
    (73, 18, 91),
    (99, 11, 110),
    (64, 36, 100),
]

@torch.no_grad()
def run_probe(dual, vanilla, tok, step):
    dual.eval(); vanilla.eval()
    d_ok = v_ok = 0
    la = dual.left.anchor.item()
    ra = dual.right.anchor.item()
    lines = [f"\n  PROBE step={step}  L-anchor={la:.4f}  R-anchor={ra:.4f}"]
    lines.append(f"  {'PROBLEM':<12} {'TARGET':<6} {'DUAL HEMI OUTPUT':<30} {'VANILLA OUTPUT':<30}")
    lines.append(f"  {'-'*82}")
    for a, b, ans in PROBE_EX:
        prompt  = f"{a}+{b}="
        ids     = tok.encode(prompt, add_bos=True, add_eos=False)
        dp_out  = gen_dual(dual, tok, ids)
        vp_out  = gen_vanilla(vanilla, tok, ids)
        dp_ans  = extract_answer(dp_out)
        vp_ans  = extract_answer(vp_out)
        dh = '✓' if dp_ans == str(ans) else '✗'
        vh = '✓' if vp_ans == str(ans) else '✗'
        if dp_ans == str(ans): d_ok += 1
        if vp_ans == str(ans): v_ok += 1
        lines.append(f"  {prompt:<12} {str(ans):<6} {dh} {dp_out:<28} {vh} {vp_out:<28}")
    lines.append(f"\n  DUAL: {d_ok}/5    VANILLA: {v_ok}/5")
    lines.append(f"  L-anchor={la:.6f}  R-anchor={ra:.6f}  (phi={PHI_CL:.6f})\n")
    msg = '\n'.join(lines)
    print(msg)
    with open(LOG_PATH,'a') as f: f.write(msg+'\n')
    sys.stdout.flush()
    dual.train(); vanilla.train()

# ============================================================
# MAIN
# ============================================================
tok     = MathTokenizer()
dual    = DualHemisphereModel(tok.vocab_size, CFG['hidden']).to(device)
vanilla = VanillaModel(tok.vocab_size, CFG['hidden']).to(device)
d_opt   = torch.optim.Adam(dual.parameters(),    lr=CFG['lr'])
v_opt   = torch.optim.Adam(vanilla.parameters(), lr=CFG['lr'])

d_params = sum(p.numel() for p in dual.parameters())
v_params = sum(p.numel() for p in vanilla.parameters())

hdr = (f"\n{'='*62}\n"
       f"  DPPU-VRU v22  |  DUAL HEMISPHERE + AGREEMENT GATE\n"
       f"  device={device}\n"
       f"  DUAL params={d_params:,}  |  Vanilla params={v_params:,}\n"
       f"\n  Left hemisphere  -- independent DPPU stream\n"
       f"  Right hemisphere -- independent DPPU stream\n"
       f"  Agreement gate   -- fires when both streams align\n"
       f"  Focus layer      -- arbitrates, outputs when agreed\n"
       f"\n  Format: ones->carry->tens->ans (v21 proven)\n"
       f"  Left anchor and Right anchor both free\n"
       f"{'='*62}\n")
print(hdr)
with open(LOG_PATH,'a') as f: f.write(hdr)

tf = CFG['tf_start']
dl = da = de = vl = va = ve = rc = 0.0

for step in range(1, CFG['max_steps']+1):
    tf = max(CFG['tf_min'], tf * CFG['tf_decay'])
    ids, starts = make_batch(CFG['batch'], tok)

    # DUAL
    dual.train()
    d_opt.zero_grad()
    d_logits = dual(ids, tf=tf)
    d_loss, d_acc = loss_and_acc(d_logits, ids, starts, tok.pad_id)
    d_loss.backward()
    torch.nn.utils.clip_grad_norm_(dual.parameters(), 5.0)
    d_opt.step()

    # VANILLA
    vanilla.train()
    v_opt.zero_grad()
    v_logits = vanilla(ids, tf=tf)
    v_loss, v_acc = loss_and_acc(v_logits, ids, starts, tok.pad_id)
    v_loss.backward()
    torch.nn.utils.clip_grad_norm_(vanilla.parameters(), 5.0)
    v_opt.step()

    dl += d_loss.item(); da += d_acc; de += mean_entropy(d_logits)
    vl += v_loss.item(); va += v_acc; ve += mean_entropy(v_logits)
    rc += 1

    if step % CFG['log_every'] == 0:
        la = dual.left.anchor.item()
        ra = dual.right.anchor.item()
        msg = (f"step={step:6d} | "
               f"DUAL  loss={dl/rc:.4f} acc={da/rc*100:.1f}% ent={de/rc:.3f} "
               f"La={la:.4f} Ra={ra:.4f} | "
               f"VANILLA  loss={vl/rc:.4f} acc={va/rc*100:.1f}% ent={ve/rc:.3f} | "
               f"tf={tf:.3f}")
        print(msg)
        with open(LOG_PATH,'a') as f: f.write(msg+'\n')
        sys.stdout.flush()
        dl = da = de = vl = va = ve = rc = 0.0

    if step % CFG['probe_every'] == 0:
        run_probe(dual, vanilla, tok, step)

print(f"\nDONE -- step={step}")
print(f"Left anchor:  {dual.left.anchor.item():.6f}")
print(f"Right anchor: {dual.right.anchor.item():.6f}")
print(f"phi=4/pi:     {PHI_CL:.6f}")

Mounted at /content/drive

  DPPU-VRU v22  |  DUAL HEMISPHERE + AGREEMENT GATE
  device=cuda
  DUAL params=772,771  |  Vanilla params=161,249

  Left hemisphere  -- independent DPPU stream
  Right hemisphere -- independent DPPU stream
  Agreement gate   -- fires when both streams align
  Focus layer      -- arbitrates, outputs when agreed

  Format: ones->carry->tens->ans (v21 proven)
  Left anchor and Right anchor both free



KeyboardInterrupt: 

In [ ]:
# ============================================================
# DPPU-VRU v22 -- DUAL HEMISPHERE + AGREEMENT GATE
# Dylan Michael Scott -- Horizon Tech
#
# Architecture inspired by bilateral cognition:
#   - Two processing streams (left/right hemisphere)
#   - Agreement gate: output only when both streams align
#   - Neutral state: hold when streams disagree
#   - Entropy tracked for shape, not just magnitude
#
# Format: step-by-step arithmetic (from v21)
#   73+18=ones:11,carry:1,tens:9,ans:91
#
# Single cell comparison: DPPU-v22 vs Vanilla
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import torch, torch.nn as nn, torch.nn.functional as F
import math, random, os, sys
from datetime import datetime

DRIVE_DIR = '/content/drive/MyDrive/dppu_vru'
LOG_PATH  = os.path.join(DRIVE_DIR, 'v22_log.txt')
os.makedirs(DRIVE_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

PI_CL  = math.pi
PHI_CL = 4.0 / math.pi
D_CAP  = 4
EPS    = 1e-7

def pi_dyn(d):  return 4.0 - (4.0 - PI_CL) * torch.exp(-d)
def phi_dyn(d): return PHI_CL * torch.exp(-d) + (1.0 - torch.exp(-d))
def omega(d):   return (pi_dyn(d) * phi_dyn(d)) / (1.0 + d + EPS)

def compute_delta(h):
    return torch.tanh(torch.abs(h) / (h.std(dim=-1, keepdim=True) + EPS))

CFG = dict(
    hidden       = 130,    # smaller -- dual arch doubles the compute
    lr           = 1e-3,
    batch        = 32,
    max_steps    = 20_000,
    log_every    = 200,
    probe_every  = 2_000,
    tf_start     = 0.9,
    tf_min       = 0.05,
    tf_decay     = 0.9995,
)

# ============================================================
# TOKENIZER
# ============================================================
class MathTokenizer:
    SPECIAL = ['<pad>', '<bos>', '<eos>', '<unk>']

    def __init__(self):
        self.vocab = {s: i for i, s in enumerate(self.SPECIAL)}
        for c in '0123456789+-*/=:,abcdefghijklmnopqrstuvwxyz_ ':
            if c not in self.vocab:
                self.vocab[c] = len(self.vocab)
        self.inv = {v: k for k, v in self.vocab.items()}
        self.pad_id = self.vocab['<pad>']
        self.bos_id = self.vocab['<bos>']
        self.eos_id = self.vocab['<eos>']
        self.eq_id  = self.vocab['=']

    @property
    def vocab_size(self): return len(self.vocab)

    def encode(self, text, add_bos=True, add_eos=True):
        return (([self.bos_id] if add_bos else []) +
                [self.vocab.get(c, self.vocab['<unk>']) for c in text] +
                ([self.eos_id] if add_eos else []))

    def decode(self, ids):
        return ''.join(self.inv.get(i,'?') for i in ids
                       if self.inv.get(i,'?') not in self.SPECIAL)

# ============================================================
# DATA -- step by step format (v21 proven)
# ============================================================
def make_step_example(a, b):
    ones     = (a % 10) + (b % 10)
    carry    = ones // 10
    tens_sum = (a // 10) + (b // 10) + carry
    return f"{a}+{b}=ones:{ones},carry:{carry},tens:{tens_sum},ans:{a+b}"

def gen_add2():
    a, b = random.randint(10,99), random.randint(10,99)
    return make_step_example(a, b)

def make_batch(bsz, tok):
    seqs, starts = [], []
    for _ in range(bsz):
        raw   = gen_add2()
        ids   = tok.encode(raw)
        start = next((j+1 for j,t in enumerate(ids) if t==tok.eq_id), 1)
        seqs.append(ids); starts.append(start)
    ml  = max(len(s) for s in seqs)
    pad = [s + [tok.pad_id]*(ml-len(s)) for s in seqs]
    return torch.tensor(pad, dtype=torch.long, device=device), starts

# ============================================================
# DPPU HEMISPHERE CELL
# One hemisphere -- half the hidden dim
# ============================================================
class DPPUHemisphere(nn.Module):
    def __init__(self, in_dim, hid):
        super().__init__()
        self.n  = D_CAP + 1
        self.ds = hid // self.n
        self.Wx = nn.Linear(in_dim, hid)
        self.Wh = nn.ModuleList([nn.Linear(hid, self.ds, bias=False) for _ in range(self.n)])
        self.Wc = nn.Linear(hid, self.n, bias=False)
        self.Wo = nn.Linear(hid, hid, bias=False)
        self.anchor = nn.Parameter(torch.tensor(PHI_CL))

    def forward(self, x, h, C):
        xp, parts = self.Wx(x), []
        for i in range(self.n):
            s, e   = i*self.ds, (i+1)*self.ds
            di     = compute_delta(h[:, s:e])
            h_rec  = (phi_dyn(di) / PHI_CL) * self.Wh[i](h)
            x_rec  = (pi_dyn(di)  / PI_CL)  * xp[:, s:e]
            gate   = torch.sigmoid(omega(di))
            anchor = C[:, i:i+1] * self.anchor
            parts.append(torch.tanh(h_rec + x_rec + anchor) * gate)
        h2 = self.Wo(torch.cat(parts, dim=-1))
        C2 = torch.clamp(torch.tanh(0.1*C + 0.01*self.Wc(h2)), -0.5, 0.5)
        return h2, C2


# ============================================================
# DUAL HEMISPHERE MODEL WITH AGREEMENT GATE
# ============================================================
class DualHemisphereModel(nn.Module):
    def __init__(self, vsz, hid=260):
        super().__init__()
        self.hid  = hid
        self.n    = D_CAP + 1
        self.emb  = nn.Embedding(vsz, hid)

        # two independent hemisphere cells
        self.left  = DPPUHemisphere(hid, hid)
        self.right = DPPUHemisphere(hid, hid)

        # agreement gate -- learns when to trust the output
        # fires when left and right are aligned
        self.agree = nn.Linear(hid * 2, hid)

        # focus layer -- the frontal arbitration
        # takes combined hemispheres + agreement signal
        self.focus = nn.Linear(hid * 2 + hid, hid)

        self.out   = nn.Linear(hid, vsz)

    def zeros(self, B):
        h = torch.zeros(B, self.hid, device=device)
        C = torch.full((B, self.n), PHI_CL, device=device)
        return h, C

    def forward(self, ids, tf=1.0):
        B, T = ids.shape
        lh, lC = self.zeros(B)
        rh, rC = self.zeros(B)
        outs = []

        for t in range(T-1):
            tok = ids[:,t] if (t==0 or random.random()<tf) else outs[-1].argmax(-1)
            x   = self.emb(tok)

            # left hemisphere processes normally
            lh, lC = self.left(x, lh, lC)

            # right hemisphere gets same input but independent state
            rh, rC = self.right(x, rh, rC)

            # agreement gate -- how aligned are the two streams?
            # dot product similarity as the agreement signal
            combined  = torch.cat([lh, rh], dim=-1)
            agreement = torch.sigmoid(self.agree(combined))  # (B, hid)

            # cosine similarity between hemispheres -- the focus signal
            cos_sim = F.cosine_similarity(lh, rh, dim=-1, eps=EPS).unsqueeze(-1)  # (B, 1)

            # focus: agreement-weighted blend of both hemispheres
            # when cos_sim high (agreement) -> strong clear signal
            # when cos_sim low (disagreement) -> muted, hold back
            focused = self.focus(torch.cat([combined, agreement], dim=-1))
            focused = focused * torch.sigmoid(cos_sim * 4.0)  # gate sharpens around 0

            outs.append(self.out(focused))

        return torch.stack(outs, dim=1)


# ============================================================
# VANILLA BASELINE
# ============================================================
class VanillaModel(nn.Module):
    def __init__(self, vsz, hid=260):
        super().__init__()
        self.hid  = hid
        self.emb  = nn.Embedding(vsz, hid)
        self.cell = nn.RNNCell(hid, hid)
        self.out  = nn.Linear(hid, vsz)

    def forward(self, ids, tf=1.0):
        B, T = ids.shape
        h    = torch.zeros(B, self.hid, device=device)
        outs = []
        for t in range(T-1):
            tok  = ids[:,t] if (t==0 or random.random()<tf) else outs[-1].argmax(-1)
            x    = self.emb(tok)
            h    = self.cell(x, h)
            outs.append(self.out(h))
        return torch.stack(outs, dim=1)

# ============================================================
# LOSS
# ============================================================
def loss_and_acc(logits, ids, starts, pad_id):
    B = logits.size(0)
    L = torch.tensor(0.0, device=logits.device)
    ok = tot = 0
    for b in range(B):
        for t in range(starts[b]-1, ids.size(1)-1):
            tgt = ids[b,t+1].item()
            if tgt == pad_id: break
            L += F.cross_entropy(logits[b,t].unsqueeze(0),
                                 torch.tensor([tgt], device=logits.device))
            if logits[b,t].argmax(-1).item() == tgt: ok += 1
            tot += 1
    return (L/tot, ok/tot) if tot>0 else (L, 0.0)

def mean_entropy(logits):
    p = torch.softmax(logits, dim=-1)
    return -(p * torch.log(p + EPS)).sum(-1).mean().item()

# ============================================================
# GENERATION
# ============================================================
@torch.no_grad()
def gen_dual(model, tok, prompt_ids, max_len=30):
    inp     = torch.tensor([prompt_ids], dtype=torch.long, device=device)
    lh, lC  = model.zeros(1)
    rh, rC  = model.zeros(1)
    for t in range(inp.size(1)-1):
        x       = model.emb(inp[:,t])
        lh, lC  = model.left(x, lh, lC)
        rh, rC  = model.right(x, rh, rC)
    cur, out = inp[:,-1], []
    for _ in range(max_len):
        x        = model.emb(cur)
        lh, lC   = model.left(x, lh, lC)
        rh, rC   = model.right(x, rh, rC)
        combined  = torch.cat([lh, rh], dim=-1)
        agreement = torch.sigmoid(model.agree(combined))
        cos_sim   = F.cosine_similarity(lh, rh, dim=-1, eps=EPS).unsqueeze(-1)
        focused   = model.focus(torch.cat([combined, agreement], dim=-1))
        focused   = focused * torch.sigmoid(cos_sim * 4.0)
        lg        = model.out(focused)
        nxt       = lg.argmax(-1)
        char      = tok.inv.get(nxt.item(), '?')
        if char in ('<eos>','<pad>'): break
        out.append(char); cur = nxt
    return ''.join(out)

@torch.no_grad()
def gen_vanilla(model, tok, prompt_ids, max_len=30):
    inp = torch.tensor([prompt_ids], dtype=torch.long, device=device)
    h   = torch.zeros(1, model.hid, device=device)
    for t in range(inp.size(1)-1):
        x = model.emb(inp[:,t])
        h = model.cell(x, h)
    cur, out = inp[:,-1], []
    for _ in range(max_len):
        x    = model.emb(cur)
        h    = model.cell(x, h)
        lg   = model.out(h)
        nxt  = lg.argmax(-1)
        char = tok.inv.get(nxt.item(), '?')
        if char in ('<eos>','<pad>'): break
        out.append(char); cur = nxt
    return ''.join(out)

def extract_answer(output):
    if 'ans:' in output:
        try: return output.split('ans:')[-1].split(',')[0].strip()
        except: pass
    return output

# ============================================================
# PROBE
# ============================================================
PROBE_EX = [
    (12, 34, 46),
    (55, 27, 82),
    (73, 18, 91),
    (99, 11, 110),
    (64, 36, 100),
]

@torch.no_grad()
def run_probe(dual, vanilla, tok, step):
    dual.eval(); vanilla.eval()
    d_ok = v_ok = 0
    la = dual.left.anchor.item()
    ra = dual.right.anchor.item()
    lines = [f"\n  PROBE step={step}  L-anchor={la:.4f}  R-anchor={ra:.4f}"]
    lines.append(f"  {'PROBLEM':<12} {'TARGET':<6} {'DUAL HEMI OUTPUT':<30} {'VANILLA OUTPUT':<30}")
    lines.append(f"  {'-'*82}")
    for a, b, ans in PROBE_EX:
        prompt  = f"{a}+{b}="
        ids     = tok.encode(prompt, add_bos=True, add_eos=False)
        dp_out  = gen_dual(dual, tok, ids)
        vp_out  = gen_vanilla(vanilla, tok, ids)
        dp_ans  = extract_answer(dp_out)
        vp_ans  = extract_answer(vp_out)
        dh = '✓' if dp_ans == str(ans) else '✗'
        vh = '✓' if vp_ans == str(ans) else '✗'
        if dp_ans == str(ans): d_ok += 1
        if vp_ans == str(ans): v_ok += 1
        lines.append(f"  {prompt:<12} {str(ans):<6} {dh} {dp_out:<28} {vh} {vp_out:<28}")
    lines.append(f"\n  DUAL: {d_ok}/5    VANILLA: {v_ok}/5")
    lines.append(f"  L-anchor={la:.6f}  R-anchor={ra:.6f}  (phi={PHI_CL:.6f})\n")
    msg = '\n'.join(lines)
    print(msg)
    with open(LOG_PATH,'a') as f: f.write(msg+'\n')
    sys.stdout.flush()
    dual.train(); vanilla.train()

# ============================================================
# MAIN
# ============================================================
tok     = MathTokenizer()
dual    = DualHemisphereModel(tok.vocab_size, CFG['hidden']).to(device)
vanilla = VanillaModel(tok.vocab_size, CFG['hidden']).to(device)
d_opt   = torch.optim.Adam(dual.parameters(),    lr=CFG['lr'])
v_opt   = torch.optim.Adam(vanilla.parameters(), lr=CFG['lr'])

d_params = sum(p.numel() for p in dual.parameters())
v_params = sum(p.numel() for p in vanilla.parameters())

hdr = (f"\n{'='*62}\n"
       f"  DPPU-VRU v22  |  DUAL HEMISPHERE + AGREEMENT GATE\n"
       f"  device={device}\n"
       f"  DUAL params={d_params:,}  |  Vanilla params={v_params:,}\n"
       f"\n  Left hemisphere  -- independent DPPU stream\n"
       f"  Right hemisphere -- independent DPPU stream\n"
       f"  Agreement gate   -- fires when both streams align\n"
       f"  Focus layer      -- arbitrates, outputs when agreed\n"
       f"\n  Format: ones->carry->tens->ans (v21 proven)\n"
       f"  Left anchor and Right anchor both free\n"
       f"{'='*62}\n")
print(hdr)
with open(LOG_PATH,'a') as f: f.write(hdr)

tf = CFG['tf_start']
dl = da = de = vl = va = ve = rc = 0.0

for step in range(1, CFG['max_steps']+1):
    tf = max(CFG['tf_min'], tf * CFG['tf_decay'])
    ids, starts = make_batch(CFG['batch'], tok)

    # DUAL
    dual.train()
    d_opt.zero_grad()
    d_logits = dual(ids, tf=tf)
    d_loss, d_acc = loss_and_acc(d_logits, ids, starts, tok.pad_id)
    d_loss.backward()
    torch.nn.utils.clip_grad_norm_(dual.parameters(), 5.0)
    d_opt.step()

    # VANILLA
    vanilla.train()
    v_opt.zero_grad()
    v_logits = vanilla(ids, tf=tf)
    v_loss, v_acc = loss_and_acc(v_logits, ids, starts, tok.pad_id)
    v_loss.backward()
    torch.nn.utils.clip_grad_norm_(vanilla.parameters(), 5.0)
    v_opt.step()

    dl += d_loss.item(); da += d_acc; de += mean_entropy(d_logits)
    vl += v_loss.item(); va += v_acc; ve += mean_entropy(v_logits)
    rc += 1

    if step % CFG['log_every'] == 0:
        la = dual.left.anchor.item()
        ra = dual.right.anchor.item()
        msg = (f"step={step:6d} | "
               f"DUAL  loss={dl/rc:.4f} acc={da/rc*100:.1f}% ent={de/rc:.3f} "
               f"La={la:.4f} Ra={ra:.4f} | "
               f"VANILLA  loss={vl/rc:.4f} acc={va/rc*100:.1f}% ent={ve/rc:.3f} | "
               f"tf={tf:.3f}")
        print(msg)
        with open(LOG_PATH,'a') as f: f.write(msg+'\n')
        sys.stdout.flush()
        dl = da = de = vl = va = ve = rc = 0.0

    if step % CFG['probe_every'] == 0:
        run_probe(dual, vanilla, tok, step)

print(f"\nDONE -- step={step}")
print(f"Left anchor:  {dual.left.anchor.item():.6f}")
print(f"Right anchor: {dual.right.anchor.item():.6f}")
print(f"phi=4/pi:     {PHI_CL:.6f}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

  DPPU-VRU v22  |  DUAL HEMISPHERE + AGREEMENT GATE
  device=cuda
  DUAL params=200,511  |  Vanilla params=46,849

  Left hemisphere  -- independent DPPU stream
  Right hemisphere -- independent DPPU stream
  Agreement gate   -- fires when both streams align
  Focus layer      -- arbitrates, outputs when agreed

  Format: ones->carry->tens->ans (v21 proven)
  Left anchor and Right anchor both free



KeyboardInterrupt: 

In [ ]:
# ============================================================
# DPPU-VRU v22 -- DUAL HEMISPHERE + AGREEMENT GATE
# Dylan Michael Scott -- Horizon Tech
#
# Architecture inspired by bilateral cognition:
#   - Two processing streams (left/right hemisphere)
#   - Agreement gate: output only when both streams align
#   - Neutral state: hold when streams disagree
#   - Entropy tracked for shape, not just magnitude
#
# Format: step-by-step arithmetic (from v21)
#   73+18=ones:11,carry:1,tens:9,ans:91
#
# Single cell comparison: DPPU-v22 vs Vanilla
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import torch, torch.nn as nn, torch.nn.functional as F
import math, random, os, sys
from datetime import datetime

DRIVE_DIR = '/content/drive/MyDrive/dppu_vru'
LOG_PATH  = os.path.join(DRIVE_DIR, 'v22_log.txt')
os.makedirs(DRIVE_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

PI_CL  = math.pi
PHI_CL = 4.0 / math.pi
D_CAP  = 4
EPS    = 1e-7

def pi_dyn(d):  return 4.0 - (4.0 - PI_CL) * torch.exp(-d)
def phi_dyn(d): return PHI_CL * torch.exp(-d) + (1.0 - torch.exp(-d))
def omega(d):   return (pi_dyn(d) * phi_dyn(d)) / (1.0 + d + EPS)

def compute_delta(h):
    return torch.tanh(torch.abs(h) / (h.std(dim=-1, keepdim=True) + EPS))

CFG = dict(
    hidden       = 130,    # smaller -- dual arch doubles the compute
    lr           = 1e-3,
    batch        = 32,
    max_steps    = 20_000,
    log_every    = 200,
    probe_every  = 2_000,
    tf_start     = 0.9,
    tf_min       = 0.05,
    tf_decay     = 0.9995,
)

# ============================================================
# TOKENIZER
# ============================================================
class MathTokenizer:
    SPECIAL = ['<pad>', '<bos>', '<eos>', '<unk>']

    def __init__(self):
        self.vocab = {s: i for i, s in enumerate(self.SPECIAL)}
        for c in '0123456789+-*/=:,abcdefghijklmnopqrstuvwxyz_ ':
            if c not in self.vocab:
                self.vocab[c] = len(self.vocab)
        self.inv = {v: k for k, v in self.vocab.items()}
        self.pad_id = self.vocab['<pad>']
        self.bos_id = self.vocab['<bos>']
        self.eos_id = self.vocab['<eos>']
        self.eq_id  = self.vocab['=']

    @property
    def vocab_size(self): return len(self.vocab)

    def encode(self, text, add_bos=True, add_eos=True):
        return (([self.bos_id] if add_bos else []) +
                [self.vocab.get(c, self.vocab['<unk>']) for c in text] +
                ([self.eos_id] if add_eos else []))

    def decode(self, ids):
        return ''.join(self.inv.get(i,'?') for i in ids
                       if self.inv.get(i,'?') not in self.SPECIAL)

# ============================================================
# DATA -- step by step format (v21 proven)
# ============================================================
def make_step_example(a, b):
    ones     = (a % 10) + (b % 10)
    carry    = ones // 10
    tens_sum = (a // 10) + (b // 10) + carry
    return f"{a}+{b}=ones:{ones},carry:{carry},tens:{tens_sum},ans:{a+b}"

def gen_add2():
    a, b = random.randint(10,99), random.randint(10,99)
    return make_step_example(a, b)

def make_batch(bsz, tok):
    seqs, starts = [], []
    for _ in range(bsz):
        raw   = gen_add2()
        ids   = tok.encode(raw)
        start = next((j+1 for j,t in enumerate(ids) if t==tok.eq_id), 1)
        seqs.append(ids); starts.append(start)
    ml  = max(len(s) for s in seqs)
    pad = [s + [tok.pad_id]*(ml-len(s)) for s in seqs]
    return torch.tensor(pad, dtype=torch.long, device=device), starts

# ============================================================
# DPPU HEMISPHERE CELL
# One hemisphere -- half the hidden dim
# ============================================================
class DPPUHemisphere(nn.Module):
    def __init__(self, in_dim, hid):
        super().__init__()
        self.n  = D_CAP + 1
        self.ds = hid // self.n
        self.hid = hid
        self.Wx  = nn.Linear(in_dim, hid)
        # single batched weight -- vectorized across dims, no Python loop
        self.Wh  = nn.Linear(hid, hid, bias=False)
        self.Wc  = nn.Linear(hid, self.n, bias=False)
        self.Wo  = nn.Linear(hid, hid, bias=False)
        self.anchor = nn.Parameter(torch.tensor(PHI_CL))

    def forward(self, x, h, C):
        B   = x.size(0)
        xp  = self.Wx(x)
        wh  = self.Wh(h)

        # reshape into (B, n, ds) -- all dim ops vectorized, no Python loop
        h_  = h.view(B, self.n, self.ds)
        xp_ = xp.view(B, self.n, self.ds)
        wh_ = wh.view(B, self.n, self.ds)

        mag    = torch.abs(h_)
        spread = h_.std(dim=-1, keepdim=True) + EPS
        di     = torch.tanh(mag / spread)

        h_rec  = (phi_dyn(di) / PHI_CL) * wh_
        x_rec  = (pi_dyn(di)  / PI_CL)  * xp_
        gate   = torch.sigmoid(omega(di))
        anchor = C.unsqueeze(-1) * self.anchor

        h_new = torch.tanh(h_rec + x_rec + anchor) * gate
        h2    = self.Wo(h_new.view(B, self.hid))
        C2    = torch.clamp(torch.tanh(0.1*C + 0.01*self.Wc(h2)), -0.5, 0.5)
        return h2, C2


# ============================================================
# DUAL HEMISPHERE MODEL WITH AGREEMENT GATE
# ============================================================
class DualHemisphereModel(nn.Module):
    def __init__(self, vsz, hid=260):
        super().__init__()
        self.hid  = hid
        self.n    = D_CAP + 1
        self.emb  = nn.Embedding(vsz, hid)

        # two independent hemisphere cells
        self.left  = DPPUHemisphere(hid, hid)
        self.right = DPPUHemisphere(hid, hid)

        # agreement gate -- learns when to trust the output
        # fires when left and right are aligned
        self.agree = nn.Linear(hid * 2, hid)

        # focus layer -- the frontal arbitration
        # takes combined hemispheres + agreement signal
        self.focus = nn.Linear(hid * 2 + hid, hid)

        self.out   = nn.Linear(hid, vsz)

    def zeros(self, B):
        h = torch.zeros(B, self.hid, device=device)
        C = torch.full((B, self.n), PHI_CL, device=device)
        return h, C

    def forward(self, ids, tf=1.0):
        B, T = ids.shape
        lh, lC = self.zeros(B)
        rh, rC = self.zeros(B)
        outs = []

        for t in range(T-1):
            tok = ids[:,t] if (t==0 or random.random()<tf) else outs[-1].argmax(-1)
            x   = self.emb(tok)

            # left hemisphere processes normally
            lh, lC = self.left(x, lh, lC)

            # right hemisphere gets same input but independent state
            rh, rC = self.right(x, rh, rC)

            # agreement gate -- how aligned are the two streams?
            # dot product similarity as the agreement signal
            combined  = torch.cat([lh, rh], dim=-1)
            agreement = torch.sigmoid(self.agree(combined))  # (B, hid)

            # cosine similarity between hemispheres -- the focus signal
            cos_sim = F.cosine_similarity(lh, rh, dim=-1, eps=EPS).unsqueeze(-1)  # (B, 1)

            # focus: agreement-weighted blend of both hemispheres
            # when cos_sim high (agreement) -> strong clear signal
            # when cos_sim low (disagreement) -> muted, hold back
            focused = self.focus(torch.cat([combined, agreement], dim=-1))
            focused = focused * torch.sigmoid(cos_sim * 4.0)  # gate sharpens around 0

            outs.append(self.out(focused))

        return torch.stack(outs, dim=1)


# ============================================================
# VANILLA BASELINE
# ============================================================
class VanillaModel(nn.Module):
    def __init__(self, vsz, hid=260):
        super().__init__()
        self.hid  = hid
        self.emb  = nn.Embedding(vsz, hid)
        self.cell = nn.RNNCell(hid, hid)
        self.out  = nn.Linear(hid, vsz)

    def forward(self, ids, tf=1.0):
        B, T = ids.shape
        h    = torch.zeros(B, self.hid, device=device)
        outs = []
        for t in range(T-1):
            tok  = ids[:,t] if (t==0 or random.random()<tf) else outs[-1].argmax(-1)
            x    = self.emb(tok)
            h    = self.cell(x, h)
            outs.append(self.out(h))
        return torch.stack(outs, dim=1)

# ============================================================
# LOSS
# ============================================================
def loss_and_acc(logits, ids, starts, pad_id):
    B = logits.size(0)
    L = torch.tensor(0.0, device=logits.device)
    ok = tot = 0
    for b in range(B):
        for t in range(starts[b]-1, ids.size(1)-1):
            tgt = ids[b,t+1].item()
            if tgt == pad_id: break
            L += F.cross_entropy(logits[b,t].unsqueeze(0),
                                 torch.tensor([tgt], device=logits.device))
            if logits[b,t].argmax(-1).item() == tgt: ok += 1
            tot += 1
    return (L/tot, ok/tot) if tot>0 else (L, 0.0)

def mean_entropy(logits):
    p = torch.softmax(logits, dim=-1)
    return -(p * torch.log(p + EPS)).sum(-1).mean().item()

# ============================================================
# GENERATION
# ============================================================
@torch.no_grad()
def gen_dual(model, tok, prompt_ids, max_len=30):
    inp     = torch.tensor([prompt_ids], dtype=torch.long, device=device)
    lh, lC  = model.zeros(1)
    rh, rC  = model.zeros(1)
    for t in range(inp.size(1)-1):
        x       = model.emb(inp[:,t])
        lh, lC  = model.left(x, lh, lC)
        rh, rC  = model.right(x, rh, rC)
    cur, out = inp[:,-1], []
    for _ in range(max_len):
        x        = model.emb(cur)
        lh, lC   = model.left(x, lh, lC)
        rh, rC   = model.right(x, rh, rC)
        combined  = torch.cat([lh, rh], dim=-1)
        agreement = torch.sigmoid(model.agree(combined))
        cos_sim   = F.cosine_similarity(lh, rh, dim=-1, eps=EPS).unsqueeze(-1)
        focused   = model.focus(torch.cat([combined, agreement], dim=-1))
        focused   = focused * torch.sigmoid(cos_sim * 4.0)
        lg        = model.out(focused)
        nxt       = lg.argmax(-1)
        char      = tok.inv.get(nxt.item(), '?')
        if char in ('<eos>','<pad>'): break
        out.append(char); cur = nxt
    return ''.join(out)

@torch.no_grad()
def gen_vanilla(model, tok, prompt_ids, max_len=30):
    inp = torch.tensor([prompt_ids], dtype=torch.long, device=device)
    h   = torch.zeros(1, model.hid, device=device)
    for t in range(inp.size(1)-1):
        x = model.emb(inp[:,t])
        h = model.cell(x, h)
    cur, out = inp[:,-1], []
    for _ in range(max_len):
        x    = model.emb(cur)
        h    = model.cell(x, h)
        lg   = model.out(h)
        nxt  = lg.argmax(-1)
        char = tok.inv.get(nxt.item(), '?')
        if char in ('<eos>','<pad>'): break
        out.append(char); cur = nxt
    return ''.join(out)

def extract_answer(output):
    if 'ans:' in output:
        try: return output.split('ans:')[-1].split(',')[0].strip()
        except: pass
    return output

# ============================================================
# PROBE
# ============================================================
PROBE_EX = [
    (12, 34, 46),
    (55, 27, 82),
    (73, 18, 91),
    (99, 11, 110),
    (64, 36, 100),
]

@torch.no_grad()
def run_probe(dual, vanilla, tok, step):
    dual.eval(); vanilla.eval()
    d_ok = v_ok = 0
    la = dual.left.anchor.item()
    ra = dual.right.anchor.item()
    lines = [f"\n  PROBE step={step}  L-anchor={la:.4f}  R-anchor={ra:.4f}"]
    lines.append(f"  {'PROBLEM':<12} {'TARGET':<6} {'DUAL HEMI OUTPUT':<30} {'VANILLA OUTPUT':<30}")
    lines.append(f"  {'-'*82}")
    for a, b, ans in PROBE_EX:
        prompt  = f"{a}+{b}="
        ids     = tok.encode(prompt, add_bos=True, add_eos=False)
        dp_out  = gen_dual(dual, tok, ids)
        vp_out  = gen_vanilla(vanilla, tok, ids)
        dp_ans  = extract_answer(dp_out)
        vp_ans  = extract_answer(vp_out)
        dh = '✓' if dp_ans == str(ans) else '✗'
        vh = '✓' if vp_ans == str(ans) else '✗'
        if dp_ans == str(ans): d_ok += 1
        if vp_ans == str(ans): v_ok += 1
        lines.append(f"  {prompt:<12} {str(ans):<6} {dh} {dp_out:<28} {vh} {vp_out:<28}")
    lines.append(f"\n  DUAL: {d_ok}/5    VANILLA: {v_ok}/5")
    lines.append(f"  L-anchor={la:.6f}  R-anchor={ra:.6f}  (phi={PHI_CL:.6f})\n")
    msg = '\n'.join(lines)
    print(msg)
    with open(LOG_PATH,'a') as f: f.write(msg+'\n')
    sys.stdout.flush()
    dual.train(); vanilla.train()

# ============================================================
# MAIN
# ============================================================
tok     = MathTokenizer()
dual    = DualHemisphereModel(tok.vocab_size, CFG['hidden']).to(device)
vanilla = VanillaModel(tok.vocab_size, CFG['hidden']).to(device)
d_opt   = torch.optim.Adam(dual.parameters(),    lr=CFG['lr'])
v_opt   = torch.optim.Adam(vanilla.parameters(), lr=CFG['lr'])

d_params = sum(p.numel() for p in dual.parameters())
v_params = sum(p.numel() for p in vanilla.parameters())

hdr = (f"\n{'='*62}\n"
       f"  DPPU-VRU v22  |  DUAL HEMISPHERE + AGREEMENT GATE\n"
       f"  device={device}\n"
       f"  DUAL params={d_params:,}  |  Vanilla params={v_params:,}\n"
       f"\n  Left hemisphere  -- independent DPPU stream\n"
       f"  Right hemisphere -- independent DPPU stream\n"
       f"  Agreement gate   -- fires when both streams align\n"
       f"  Focus layer      -- arbitrates, outputs when agreed\n"
       f"\n  Format: ones->carry->tens->ans (v21 proven)\n"
       f"  Left anchor and Right anchor both free\n"
       f"{'='*62}\n")
print(hdr)
with open(LOG_PATH,'a') as f: f.write(hdr)

tf = CFG['tf_start']
dl = da = de = vl = va = ve = rc = 0.0

for step in range(1, CFG['max_steps']+1):
    tf = max(CFG['tf_min'], tf * CFG['tf_decay'])
    ids, starts = make_batch(CFG['batch'], tok)

    # DUAL
    dual.train()
    d_opt.zero_grad()
    d_logits = dual(ids, tf=tf)
    d_loss, d_acc = loss_and_acc(d_logits, ids, starts, tok.pad_id)
    d_loss.backward()
    torch.nn.utils.clip_grad_norm_(dual.parameters(), 5.0)
    d_opt.step()

    # VANILLA
    vanilla.train()
    v_opt.zero_grad()
    v_logits = vanilla(ids, tf=tf)
    v_loss, v_acc = loss_and_acc(v_logits, ids, starts, tok.pad_id)
    v_loss.backward()
    torch.nn.utils.clip_grad_norm_(vanilla.parameters(), 5.0)
    v_opt.step()

    dl += d_loss.item(); da += d_acc; de += mean_entropy(d_logits)
    vl += v_loss.item(); va += v_acc; ve += mean_entropy(v_logits)
    rc += 1

    if step % CFG['log_every'] == 0:
        la = dual.left.anchor.item()
        ra = dual.right.anchor.item()
        msg = (f"step={step:6d} | "
               f"DUAL  loss={dl/rc:.4f} acc={da/rc*100:.1f}% ent={de/rc:.3f} "
               f"La={la:.4f} Ra={ra:.4f} | "
               f"VANILLA  loss={vl/rc:.4f} acc={va/rc*100:.1f}% ent={ve/rc:.3f} | "
               f"tf={tf:.3f}")
        print(msg)
        with open(LOG_PATH,'a') as f: f.write(msg+'\n')
        sys.stdout.flush()
        dl = da = de = vl = va = ve = rc = 0.0

    if step % CFG['probe_every'] == 0:
        run_probe(dual, vanilla, tok, step)

print(f"\nDONE -- step={step}")
print(f"Left anchor:  {dual.left.anchor.item():.6f}")
print(f"Right anchor: {dual.right.anchor.item():.6f}")
print(f"phi=4/pi:     {PHI_CL:.6f}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

  DPPU-VRU v22  |  DUAL HEMISPHERE + AGREEMENT GATE
  device=cuda
  DUAL params=200,511  |  Vanilla params=46,849

  Left hemisphere  -- independent DPPU stream
  Right hemisphere -- independent DPPU stream
  Agreement gate   -- fires when both streams align
  Focus layer      -- arbitrates, outputs when agreed

  Format: ones->carry->tens->ans (v21 proven)
  Left anchor and Right anchor both free



KeyboardInterrupt: 

In [ ]:
# ============================================================
# DPPU-VRU v22 -- DUAL HEMISPHERE + AGREEMENT GATE
# Dylan Michael Scott -- Horizon Tech
#
# Two independent DPPU streams (hemispheres)
# Agreement gate fires when both streams align
# Focus layer arbitrates the output
# Per-hemisphere probe so we can see what each side is doing
#
# Vectorized cell and loss -- no Python loops, runs fast
# Format: ones->carry->tens->ans (v21 proven)
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import torch, torch.nn as nn, torch.nn.functional as F
import math, random, os, sys
from datetime import datetime

DRIVE_DIR = '/content/drive/MyDrive/dppu_vru'
LOG_PATH  = os.path.join(DRIVE_DIR, 'v22_log.txt')
os.makedirs(DRIVE_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

PI_CL  = math.pi
PHI_CL = 4.0 / math.pi
D_CAP  = 4
EPS    = 1e-7

def pi_dyn(d):  return 4.0 - (4.0 - PI_CL) * torch.exp(-d)
def phi_dyn(d): return PHI_CL * torch.exp(-d) + (1.0 - torch.exp(-d))
def omega(d):   return (pi_dyn(d) * phi_dyn(d)) / (1.0 + d + EPS)

CFG = dict(
    hidden      = 130,     # divisible by 5; dual arch doubles compute
    lr          = 1e-3,
    batch       = 32,
    max_steps   = 20_000,
    log_every   = 200,
    probe_every = 2_000,
    tf_start    = 0.9,
    tf_min      = 0.05,
    tf_decay    = 0.9995,
)

# ============================================================
# TOKENIZER
# ============================================================
class MathTokenizer:
    SPECIAL = ['<pad>', '<bos>', '<eos>', '<unk>']
    def __init__(self):
        self.vocab = {s: i for i, s in enumerate(self.SPECIAL)}
        for c in '0123456789+-*/=:,abcdefghijklmnopqrstuvwxyz_ ':
            if c not in self.vocab:
                self.vocab[c] = len(self.vocab)
        self.inv    = {v: k for k, v in self.vocab.items()}
        self.pad_id = self.vocab['<pad>']
        self.bos_id = self.vocab['<bos>']
        self.eos_id = self.vocab['<eos>']
        self.eq_id  = self.vocab['=']
    @property
    def vocab_size(self): return len(self.vocab)
    def encode(self, text, add_bos=True, add_eos=True):
        return (([self.bos_id] if add_bos else []) +
                [self.vocab.get(c, self.vocab['<unk>']) for c in text] +
                ([self.eos_id] if add_eos else []))
    def decode(self, ids):
        return ''.join(self.inv.get(i,'?') for i in ids
                       if self.inv.get(i,'?') not in self.SPECIAL)

# ============================================================
# DATA
# ============================================================
def make_step_example(a, b):
    ones     = (a % 10) + (b % 10)
    carry    = ones // 10
    tens_sum = (a // 10) + (b // 10) + carry
    return f"{a}+{b}=ones:{ones},carry:{carry},tens:{tens_sum},ans:{a+b}"

def gen_add2():
    a, b = random.randint(10,99), random.randint(10,99)
    return make_step_example(a, b)

def make_batch(bsz, tok):
    seqs, starts = [], []
    for _ in range(bsz):
        raw   = gen_add2()
        ids   = tok.encode(raw)
        start = next((j+1 for j,t in enumerate(ids) if t==tok.eq_id), 1)
        seqs.append(ids); starts.append(start)
    ml  = max(len(s) for s in seqs)
    pad = [s + [tok.pad_id]*(ml-len(s)) for s in seqs]
    return torch.tensor(pad, dtype=torch.long, device=device), starts

# ============================================================
# DPPU HEMISPHERE CELL -- fully vectorized, no Python loop
# ============================================================
class DPPUHemisphere(nn.Module):
    def __init__(self, in_dim, hid):
        super().__init__()
        self.n   = D_CAP + 1
        self.ds  = hid // self.n
        self.hid = hid
        self.Wx  = nn.Linear(in_dim, hid)
        self.Wh  = nn.Linear(hid, hid, bias=False)
        self.Wc  = nn.Linear(hid, self.n, bias=False)
        self.Wo  = nn.Linear(hid, hid, bias=False)
        self.anchor = nn.Parameter(torch.tensor(PHI_CL))

    def forward(self, x, h, C):
        B  = x.size(0)
        xp = self.Wx(x)
        wh = self.Wh(h)
        # reshape to (B, n, ds) -- all ops vectorized across dims
        h_  = h.view(B, self.n, self.ds)
        xp_ = xp.view(B, self.n, self.ds)
        wh_ = wh.view(B, self.n, self.ds)
        # delta -- geometric state of each dim
        di  = torch.tanh(torch.abs(h_) / (h_.std(dim=-1, keepdim=True) + EPS))
        # dynamic operators
        h_rec  = (phi_dyn(di) / PHI_CL) * wh_
        x_rec  = (pi_dyn(di)  / PI_CL)  * xp_
        gate   = torch.sigmoid(omega(di))
        anchor = C.unsqueeze(-1) * self.anchor
        h_new  = torch.tanh(h_rec + x_rec + anchor) * gate
        h2 = self.Wo(h_new.view(B, self.hid))
        C2 = torch.clamp(torch.tanh(0.1*C + 0.01*self.Wc(h2)), -0.5, 0.5)
        return h2, C2

# ============================================================
# DUAL HEMISPHERE MODEL
# ============================================================
class DualHemisphereModel(nn.Module):
    def __init__(self, vsz, hid=130):
        super().__init__()
        self.hid   = hid
        self.n     = D_CAP + 1
        self.emb   = nn.Embedding(vsz, hid)
        self.left  = DPPUHemisphere(hid, hid)
        self.right = DPPUHemisphere(hid, hid)
        # agreement gate: do both hemispheres agree?
        self.agree = nn.Linear(hid * 2, hid)
        # focus: frontal arbitration layer
        self.focus = nn.Linear(hid * 2 + hid, hid)
        self.out   = nn.Linear(hid, vsz)

    def zeros(self, B):
        h = torch.zeros(B, self.hid, device=device)
        C = torch.full((B, self.n), PHI_CL, device=device)
        return h, C

    def forward(self, ids, tf=1.0):
        B, T = ids.shape
        lh, lC = self.zeros(B)
        rh, rC = self.zeros(B)
        outs = []
        for t in range(T-1):
            tok = ids[:,t] if (t==0 or random.random()<tf) else outs[-1].argmax(-1)
            x   = self.emb(tok)
            lh, lC = self.left(x,  lh, lC)
            rh, rC = self.right(x, rh, rC)
            combined  = torch.cat([lh, rh], dim=-1)
            agreement = torch.sigmoid(self.agree(combined))
            cos_sim   = F.cosine_similarity(lh, rh, dim=-1, eps=EPS).unsqueeze(-1)
            focused   = self.focus(torch.cat([combined, agreement], dim=-1))
            focused   = focused * torch.sigmoid(cos_sim * 4.0)
            outs.append(self.out(focused))
        return torch.stack(outs, dim=1)

# ============================================================
# VANILLA BASELINE
# ============================================================
class VanillaModel(nn.Module):
    def __init__(self, vsz, hid=130):
        super().__init__()
        self.hid  = hid
        self.emb  = nn.Embedding(vsz, hid)
        self.cell = nn.RNNCell(hid, hid)
        self.out  = nn.Linear(hid, vsz)

    def forward(self, ids, tf=1.0):
        B, T = ids.shape
        h    = torch.zeros(B, self.hid, device=device)
        outs = []
        for t in range(T-1):
            tok  = ids[:,t] if (t==0 or random.random()<tf) else outs[-1].argmax(-1)
            x    = self.emb(tok)
            h    = self.cell(x, h)
            outs.append(self.out(h))
        return torch.stack(outs, dim=1)

# ============================================================
# LOSS -- fully vectorized
# ============================================================
def loss_and_acc(logits, ids, starts, pad_id):
    B, T, V = logits.shape
    targets = ids[:, 1:]                          # (B, T-1)
    mask    = (targets != pad_id)
    for b in range(B):                            # mask question tokens
        if starts[b]-1 > 0:
            mask[b, :starts[b]-1] = False
    flat_logits  = logits.reshape(-1, V)
    flat_targets = targets.reshape(-1)
    flat_mask    = mask.reshape(-1)
    if flat_mask.sum() == 0:
        return torch.tensor(0.0, device=logits.device), 0.0
    loss = F.cross_entropy(flat_logits[flat_mask], flat_targets[flat_mask])
    acc  = (flat_logits[flat_mask].argmax(-1) == flat_targets[flat_mask]).float().mean().item()
    return loss, acc

def mean_entropy(logits):
    p = torch.softmax(logits, dim=-1)
    return -(p * torch.log(p + EPS)).sum(-1).mean().item()

# ============================================================
# GENERATION
# ============================================================
@torch.no_grad()
def gen_dual(model, tok, prompt_ids, max_len=30):
    inp    = torch.tensor([prompt_ids], dtype=torch.long, device=device)
    lh, lC = model.zeros(1)
    rh, rC = model.zeros(1)
    for t in range(inp.size(1)-1):
        x      = model.emb(inp[:,t])
        lh, lC = model.left(x,  lh, lC)
        rh, rC = model.right(x, rh, rC)
    cur, out = inp[:,-1], []
    for _ in range(max_len):
        x         = model.emb(cur)
        lh, lC    = model.left(x,  lh, lC)
        rh, rC    = model.right(x, rh, rC)
        combined  = torch.cat([lh, rh], dim=-1)
        agreement = torch.sigmoid(model.agree(combined))
        cos_sim   = F.cosine_similarity(lh, rh, dim=-1, eps=EPS).unsqueeze(-1)
        focused   = model.focus(torch.cat([combined, agreement], dim=-1))
        focused   = focused * torch.sigmoid(cos_sim * 4.0)
        lg  = model.out(focused)
        nxt = lg.argmax(-1)
        char = tok.inv.get(nxt.item(), '?')
        if char in ('<eos>','<pad>'): break
        out.append(char); cur = nxt
    return ''.join(out)

@torch.no_grad()
def gen_hemi(model, tok, prompt_ids, side='left', max_len=30):
    # one hemisphere only -- bypasses agreement gate
    inp    = torch.tensor([prompt_ids], dtype=torch.long, device=device)
    cell   = model.left if side == 'left' else model.right
    h, C   = model.zeros(1)
    for t in range(inp.size(1)-1):
        x    = model.emb(inp[:,t])
        h, C = cell(x, h, C)
    cur, out = inp[:,-1], []
    for _ in range(max_len):
        x    = model.emb(cur)
        h, C = cell(x, h, C)
        lg   = model.out(h)
        nxt  = lg.argmax(-1)
        char = tok.inv.get(nxt.item(), '?')
        if char in ('<eos>','<pad>'): break
        out.append(char); cur = nxt
    return ''.join(out)

@torch.no_grad()
def gen_vanilla(model, tok, prompt_ids, max_len=30):
    inp = torch.tensor([prompt_ids], dtype=torch.long, device=device)
    h   = torch.zeros(1, model.hid, device=device)
    for t in range(inp.size(1)-1):
        x = model.emb(inp[:,t])
        h = model.cell(x, h)
    cur, out = inp[:,-1], []
    for _ in range(max_len):
        x    = model.emb(cur)
        h    = model.cell(x, h)
        lg   = model.out(h)
        nxt  = lg.argmax(-1)
        char = tok.inv.get(nxt.item(), '?')
        if char in ('<eos>','<pad>'): break
        out.append(char); cur = nxt
    return ''.join(out)

def extract_answer(output):
    if 'ans:' in output:
        try: return output.split('ans:')[-1].split(',')[0].strip()
        except: pass
    return output

# ============================================================
# PROBE -- dual full output + each hemisphere + vanilla
# ============================================================
PROBE_EX = [(12,34,46),(55,27,82),(73,18,91),(99,11,110),(64,36,100)]

@torch.no_grad()
def run_probe(dual, vanilla, tok, step):
    dual.eval(); vanilla.eval()
    d_ok = v_ok = 0
    la  = dual.left.anchor.item()
    ra  = dual.right.anchor.item()
    agw = dual.agree.weight.abs().mean().item()
    fow = dual.focus.weight.abs().mean().item()
    lines = [
        f"\n  PROBE step={step}",
        f"  L-anchor={la:.6f}  R-anchor={ra:.6f}  phi={PHI_CL:.6f}",
        f"  agree_gate={agw:.4f}  focus={fow:.4f}",
        f"  {'PROB':<10} {'ANS':<5} {'DUAL':<24} {'LEFT':<20} {'RIGHT':<20} {'VANILLA':<20}",
        f"  {'-'*95}",
    ]
    for a, b, ans in PROBE_EX:
        prompt = f"{a}+{b}="
        ids    = tok.encode(prompt, add_bos=True, add_eos=False)
        dp     = gen_dual(dual, tok, ids)
        lo     = gen_hemi(dual, tok, ids, 'left')
        ro     = gen_hemi(dual, tok, ids, 'right')
        vp     = gen_vanilla(vanilla, tok, ids)
        dp_ans = extract_answer(dp)
        dh = '✓' if dp_ans == str(ans) else '✗'
        vh = '✓' if extract_answer(vp) == str(ans) else '✗'
        if dp_ans == str(ans): d_ok += 1
        if extract_answer(vp) == str(ans): v_ok += 1
        lines.append(f"  {prompt:<10} {str(ans):<5} {dh} {dp:<22} {lo:<20} {ro:<20} {vh} {vp}")
    lines.append(f"\n  DUAL: {d_ok}/5   VANILLA: {v_ok}/5\n")
    msg = '\n'.join(lines)
    print(msg)
    with open(LOG_PATH,'a') as f: f.write(msg+'\n')
    sys.stdout.flush()
    dual.train(); vanilla.train()

# ============================================================
# MAIN
# ============================================================
tok     = MathTokenizer()
dual    = DualHemisphereModel(tok.vocab_size, CFG['hidden']).to(device)
vanilla = VanillaModel(tok.vocab_size, CFG['hidden']).to(device)
d_opt   = torch.optim.Adam(dual.parameters(),    lr=CFG['lr'])
v_opt   = torch.optim.Adam(vanilla.parameters(), lr=CFG['lr'])

d_params = sum(p.numel() for p in dual.parameters())
v_params = sum(p.numel() for p in vanilla.parameters())

hdr = (f"\n{'='*62}\n"
       f"  DPPU-VRU v22  |  DUAL HEMISPHERE + AGREEMENT GATE\n"
       f"  device={device}\n"
       f"  DUAL params={d_params:,}  |  Vanilla params={v_params:,}\n"
       f"\n  Left hemisphere  -- independent DPPU stream\n"
       f"  Right hemisphere -- independent DPPU stream\n"
       f"  Agreement gate   -- fires when both streams align\n"
       f"  Focus layer      -- arbitrates, outputs when agreed\n"
       f"  Probe shows: dual | left-only | right-only | vanilla\n"
       f"\n  Format: ones->carry->tens->ans (v21 proven)\n"
       f"  Both anchors free -- no regularization\n"
       f"{'='*62}\n")
print(hdr)
with open(LOG_PATH,'a') as f: f.write(hdr)

tf = CFG['tf_start']
dl = da = de = vl = va = ve = rc = 0.0

for step in range(1, CFG['max_steps']+1):
    tf = max(CFG['tf_min'], tf * CFG['tf_decay'])
    ids, starts = make_batch(CFG['batch'], tok)

    dual.train()
    d_opt.zero_grad()
    d_logits = dual(ids, tf=tf)
    d_loss, d_acc = loss_and_acc(d_logits, ids, starts, tok.pad_id)
    d_loss.backward()
    torch.nn.utils.clip_grad_norm_(dual.parameters(), 5.0)
    d_opt.step()

    vanilla.train()
    v_opt.zero_grad()
    v_logits = vanilla(ids, tf=tf)
    v_loss, v_acc = loss_and_acc(v_logits, ids, starts, tok.pad_id)
    v_loss.backward()
    torch.nn.utils.clip_grad_norm_(vanilla.parameters(), 5.0)
    v_opt.step()

    dl += d_loss.item(); da += d_acc; de += mean_entropy(d_logits)
    vl += v_loss.item(); va += v_acc; ve += mean_entropy(v_logits)
    rc += 1

    if step % CFG['log_every'] == 0:
        la = dual.left.anchor.item()
        ra = dual.right.anchor.item()
        msg = (f"step={step:6d} | "
               f"DUAL loss={dl/rc:.4f} acc={da/rc*100:.1f}% ent={de/rc:.3f} "
               f"La={la:.4f} Ra={ra:.4f} | "
               f"VANILLA loss={vl/rc:.4f} acc={va/rc*100:.1f}% ent={ve/rc:.3f} | "
               f"tf={tf:.3f}")
        print(msg)
        with open(LOG_PATH,'a') as f: f.write(msg+'\n')
        sys.stdout.flush()
        dl = da = de = vl = va = ve = rc = 0.0

    if step % CFG['probe_every'] == 0:
        run_probe(dual, vanilla, tok, step)

print(f"\nDONE -- step={step}")
print(f"Left anchor:  {dual.left.anchor.item():.6f}")
print(f"Right anchor: {dual.right.anchor.item():.6f}")
print(f"phi=4/pi:     {PHI_CL:.6f}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

  DPPU-VRU v22  |  DUAL HEMISPHERE + AGREEMENT GATE
  device=cuda
  DUAL params=200,511  |  Vanilla params=46,849

  Left hemisphere  -- independent DPPU stream
  Right hemisphere -- independent DPPU stream
  Agreement gate   -- fires when both streams align
  Focus layer      -- arbitrates, outputs when agreed
  Probe shows: dual | left-only | right-only | vanilla

  Format: ones->carry->tens->ans (v21 proven)
  Both anchors free -- no regularization



KeyboardInterrupt: 

In [ ]:
# ============================================================
# DPPU-VRU v22 -- DUAL HEMISPHERE + AGREEMENT GATE
# Dylan Michael Scott -- Horizon Tech
#
# Two independent DPPU streams (hemispheres)
# Agreement gate fires when both streams align
# Focus layer arbitrates the output
# Per-hemisphere probe so we can see what each side is doing
#
# Vectorized cell and loss -- no Python loops, runs fast
# Format: ones->carry->tens->ans (v21 proven)
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import torch, torch.nn as nn, torch.nn.functional as F
import math, random, os, sys
from datetime import datetime

DRIVE_DIR = '/content/drive/MyDrive/dppu_vru'
LOG_PATH  = os.path.join(DRIVE_DIR, 'v22_log.txt')
os.makedirs(DRIVE_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

PI_CL  = math.pi
PHI_CL = 4.0 / math.pi
D_CAP  = 4
EPS    = 1e-7

def pi_dyn(d):  return 4.0 - (4.0 - PI_CL) * torch.exp(-d)
def phi_dyn(d): return PHI_CL * torch.exp(-d) + (1.0 - torch.exp(-d))
def omega(d):   return (pi_dyn(d) * phi_dyn(d)) / (1.0 + d + EPS)

CFG = dict(
    hidden      = 130,
    lr          = 1e-3,
    batch       = 32,
    max_steps   = 20_000,
    log_every   = 200,
    probe_every = 2_000,
    tf_start    = 0.9,
    tf_min      = 0.05,
    tf_decay    = 0.9995,
)

class MathTokenizer:
    SPECIAL = ['<pad>', '<bos>', '<eos>', '<unk>']
    def __init__(self):
        self.vocab = {s: i for i, s in enumerate(self.SPECIAL)}
        for c in '0123456789+-*/=:,abcdefghijklmnopqrstuvwxyz_ ':
            if c not in self.vocab:
                self.vocab[c] = len(self.vocab)
        self.inv    = {v: k for k, v in self.vocab.items()}
        self.pad_id = self.vocab['<pad>']
        self.bos_id = self.vocab['<bos>']
        self.eos_id = self.vocab['<eos>']
        self.eq_id  = self.vocab['=']
    @property
    def vocab_size(self): return len(self.vocab)
    def encode(self, text, add_bos=True, add_eos=True):
        return (([self.bos_id] if add_bos else []) +
                [self.vocab.get(c, self.vocab['<unk>']) for c in text] +
                ([self.eos_id] if add_eos else []))
    def decode(self, ids):
        return ''.join(self.inv.get(i,'?') for i in ids
                       if self.inv.get(i,'?') not in self.SPECIAL)

def make_step_example(a, b):
    ones     = (a % 10) + (b % 10)
    carry    = ones // 10
    tens_sum = (a // 10) + (b // 10) + carry
    return f"{a}+{b}=ones:{ones},carry:{carry},tens:{tens_sum},ans:{a+b}"

def gen_add2():
    a, b = random.randint(10,99), random.randint(10,99)
    return make_step_example(a, b)

def make_batch(bsz, tok):
    seqs, starts = [], []
    for _ in range(bsz):
        raw   = gen_add2()
        ids   = tok.encode(raw)
        start = next((j+1 for j,t in enumerate(ids) if t==tok.eq_id), 1)
        seqs.append(ids); starts.append(start)
    ml  = max(len(s) for s in seqs)
    pad = [s + [tok.pad_id]*(ml-len(s)) for s in seqs]
    return torch.tensor(pad, dtype=torch.long, device=device), starts

class DPPUHemisphere(nn.Module):
    def __init__(self, in_dim, hid):
        super().__init__()
        self.n   = D_CAP + 1
        self.ds  = hid // self.n
        self.hid = hid
        self.Wx  = nn.Linear(in_dim, hid)
        self.Wh  = nn.Linear(hid, hid, bias=False)
        self.Wc  = nn.Linear(hid, self.n, bias=False)
        self.Wo  = nn.Linear(hid, hid, bias=False)
        self.anchor = nn.Parameter(torch.tensor(PHI_CL))

    def forward(self, x, h, C):
        B  = x.size(0)
        xp = self.Wx(x)
        wh = self.Wh(h)
        h_  = h.view(B, self.n, self.ds)
        xp_ = xp.view(B, self.n, self.ds)
        wh_ = wh.view(B, self.n, self.ds)
        di     = torch.tanh(torch.abs(h_) / (h_.std(dim=-1, keepdim=True) + EPS))
        h_rec  = (phi_dyn(di) / PHI_CL) * wh_
        x_rec  = (pi_dyn(di)  / PI_CL)  * xp_
        gate   = torch.sigmoid(omega(di))
        anchor = C.unsqueeze(-1) * self.anchor
        h_new  = torch.tanh(h_rec + x_rec + anchor) * gate
        h2 = self.Wo(h_new.view(B, self.hid))
        C2 = torch.clamp(torch.tanh(0.1*C + 0.01*self.Wc(h2)), -0.5, 0.5)
        return h2, C2

class DualHemisphereModel(nn.Module):
    def __init__(self, vsz, hid=130):
        super().__init__()
        self.hid   = hid
        self.n     = D_CAP + 1
        self.emb   = nn.Embedding(vsz, hid)
        self.left  = DPPUHemisphere(hid, hid)
        self.right = DPPUHemisphere(hid, hid)
        self.agree = nn.Linear(hid * 2, hid)
        self.focus = nn.Linear(hid * 2 + hid, hid)
        self.out   = nn.Linear(hid, vsz)

    def zeros(self, B):
        h = torch.zeros(B, self.hid, device=device)
        C = torch.full((B, self.n), PHI_CL, device=device)
        return h, C

    def forward(self, ids, tf=1.0, tbptt=8):
        B, T = ids.shape
        lh, lC = self.zeros(B)
        rh, rC = self.zeros(B)
        outs = []
        for t in range(T-1):
            tok = ids[:,t] if (t==0 or random.random()<tf) else outs[-1].argmax(-1)
            x   = self.emb(tok)
            # detach every tbptt steps -- truncated BPTT, keeps backward fast
            if t > 0 and t % tbptt == 0:
                lh, lC = lh.detach(), lC.detach()
                rh, rC = rh.detach(), rC.detach()
            lh, lC = self.left(x,  lh, lC)
            rh, rC = self.right(x, rh, rC)
            combined  = torch.cat([lh, rh], dim=-1)
            agreement = torch.sigmoid(self.agree(combined))
            cos_sim   = F.cosine_similarity(lh, rh, dim=-1, eps=EPS).unsqueeze(-1)
            focused   = self.focus(torch.cat([combined, agreement], dim=-1))
            focused   = focused * torch.sigmoid(cos_sim * 4.0)
            outs.append(self.out(focused))
        return torch.stack(outs, dim=1)

class VanillaModel(nn.Module):
    def __init__(self, vsz, hid=130):
        super().__init__()
        self.hid  = hid
        self.emb  = nn.Embedding(vsz, hid)
        self.cell = nn.RNNCell(hid, hid)
        self.out  = nn.Linear(hid, vsz)

    def forward(self, ids, tf=1.0):
        B, T = ids.shape
        h    = torch.zeros(B, self.hid, device=device)
        outs = []
        for t in range(T-1):
            tok  = ids[:,t] if (t==0 or random.random()<tf) else outs[-1].argmax(-1)
            x    = self.emb(tok)
            h    = self.cell(x, h)
            outs.append(self.out(h))
        return torch.stack(outs, dim=1)

def loss_and_acc(logits, ids, starts, pad_id):
    B, T, V = logits.shape
    targets = ids[:, 1:]
    mask    = (targets != pad_id)
    for b in range(B):
        if starts[b]-1 > 0:
            mask[b, :starts[b]-1] = False
    fl = logits.reshape(-1, V)
    ft = targets.reshape(-1)
    fm = mask.reshape(-1)
    if fm.sum() == 0:
        return torch.tensor(0.0, device=logits.device), 0.0
    loss = F.cross_entropy(fl[fm], ft[fm])
    acc  = (fl[fm].argmax(-1) == ft[fm]).float().mean().item()
    return loss, acc

def mean_entropy(logits):
    p = torch.softmax(logits, dim=-1)
    return -(p * torch.log(p + EPS)).sum(-1).mean().item()

@torch.no_grad()
def gen_dual(model, tok, prompt_ids, max_len=30):
    inp    = torch.tensor([prompt_ids], dtype=torch.long, device=device)
    lh, lC = model.zeros(1)
    rh, rC = model.zeros(1)
    for t in range(inp.size(1)-1):
        x      = model.emb(inp[:,t])
        lh, lC = model.left(x,  lh, lC)
        rh, rC = model.right(x, rh, rC)
    cur, out = inp[:,-1], []
    for _ in range(max_len):
        x         = model.emb(cur)
        lh, lC    = model.left(x,  lh, lC)
        rh, rC    = model.right(x, rh, rC)
        combined  = torch.cat([lh, rh], dim=-1)
        agreement = torch.sigmoid(model.agree(combined))
        cos_sim   = F.cosine_similarity(lh, rh, dim=-1, eps=EPS).unsqueeze(-1)
        focused   = model.focus(torch.cat([combined, agreement], dim=-1))
        focused   = focused * torch.sigmoid(cos_sim * 4.0)
        lg   = model.out(focused)
        nxt  = lg.argmax(-1)
        char = tok.inv.get(nxt.item(), '?')
        if char in ('<eos>','<pad>'): break
        out.append(char); cur = nxt
    return ''.join(out)

@torch.no_grad()
def gen_hemi(model, tok, prompt_ids, side='left', max_len=30):
    inp    = torch.tensor([prompt_ids], dtype=torch.long, device=device)
    cell   = model.left if side == 'left' else model.right
    h, C   = model.zeros(1)
    for t in range(inp.size(1)-1):
        x    = model.emb(inp[:,t])
        h, C = cell(x, h, C)
    cur, out = inp[:,-1], []
    for _ in range(max_len):
        x    = model.emb(cur)
        h, C = cell(x, h, C)
        lg   = model.out(h)
        nxt  = lg.argmax(-1)
        char = tok.inv.get(nxt.item(), '?')
        if char in ('<eos>','<pad>'): break
        out.append(char); cur = nxt
    return ''.join(out)

@torch.no_grad()
def gen_vanilla(model, tok, prompt_ids, max_len=30):
    inp = torch.tensor([prompt_ids], dtype=torch.long, device=device)
    h   = torch.zeros(1, model.hid, device=device)
    for t in range(inp.size(1)-1):
        x = model.emb(inp[:,t])
        h = model.cell(x, h)
    cur, out = inp[:,-1], []
    for _ in range(max_len):
        x    = model.emb(cur)
        h    = model.cell(x, h)
        lg   = model.out(h)
        nxt  = lg.argmax(-1)
        char = tok.inv.get(nxt.item(), '?')
        if char in ('<eos>','<pad>'): break
        out.append(char); cur = nxt
    return ''.join(out)

def extract_answer(output):
    if 'ans:' in output:
        try: return output.split('ans:')[-1].split(',')[0].strip()
        except: pass
    return output

PROBE_EX = [(12,34,46),(55,27,82),(73,18,91),(99,11,110),(64,36,100)]

@torch.no_grad()
def run_probe(dual, vanilla, tok, step):
    dual.eval(); vanilla.eval()
    d_ok = v_ok = 0
    la  = dual.left.anchor.item()
    ra  = dual.right.anchor.item()
    agw = dual.agree.weight.abs().mean().item()
    fow = dual.focus.weight.abs().mean().item()
    lines = [
        f"\n  PROBE step={step}",
        f"  L-anchor={la:.6f}  R-anchor={ra:.6f}  phi={PHI_CL:.6f}",
        f"  agree_gate={agw:.4f}  focus={fow:.4f}",
        f"  {'PROB':<10} {'ANS':<5} {'DUAL OUTPUT':<26} {'LEFT':<22} {'RIGHT':<22} {'VANILLA':<18}",
        f"  {'-'*100}",
    ]
    for a, b, ans in PROBE_EX:
        prompt = f"{a}+{b}="
        ids    = tok.encode(prompt, add_bos=True, add_eos=False)
        dp     = gen_dual(dual, tok, ids)
        lo     = gen_hemi(dual, tok, ids, 'left')
        ro     = gen_hemi(dual, tok, ids, 'right')
        vp     = gen_vanilla(vanilla, tok, ids)
        dp_ans = extract_answer(dp)
        dh = '✓' if dp_ans == str(ans) else '✗'
        vh = '✓' if extract_answer(vp) == str(ans) else '✗'
        if dp_ans == str(ans): d_ok += 1
        if extract_answer(vp) == str(ans): v_ok += 1
        lines.append(f"  {prompt:<10} {str(ans):<5} {dh} {dp:<24} {lo:<22} {ro:<22} {vh} {vp}")
    lines.append(f"\n  DUAL: {d_ok}/5   VANILLA: {v_ok}/5\n")
    msg = '\n'.join(lines)
    print(msg)
    with open(LOG_PATH,'a') as f: f.write(msg+'\n')
    sys.stdout.flush()
    dual.train(); vanilla.train()

# ============================================================
# MAIN
# ============================================================
tok     = MathTokenizer()
dual    = DualHemisphereModel(tok.vocab_size, CFG['hidden']).to(device)
vanilla = VanillaModel(tok.vocab_size, CFG['hidden']).to(device)
d_opt   = torch.optim.Adam(dual.parameters(),    lr=CFG['lr'])
v_opt   = torch.optim.Adam(vanilla.parameters(), lr=CFG['lr'])

d_params = sum(p.numel() for p in dual.parameters())
v_params = sum(p.numel() for p in vanilla.parameters())

hdr = (f"\n{'='*62}\n"
       f"  DPPU-VRU v22  |  DUAL HEMISPHERE + AGREEMENT GATE\n"
       f"  device={device}\n"
       f"  DUAL params={d_params:,}  |  Vanilla params={v_params:,}\n"
       f"\n  Left hemisphere  -- independent DPPU stream\n"
       f"  Right hemisphere -- independent DPPU stream\n"
       f"  Agreement gate   -- fires when both streams align\n"
       f"  Focus layer      -- arbitrates, outputs when agreed\n"
       f"  Probe: dual | left-only | right-only | vanilla\n"
       f"\n  Format: ones->carry->tens->ans (v21 proven)\n"
       f"  Both anchors free -- no regularization\n"
       f"{'='*62}\n")
print(hdr)
with open(LOG_PATH,'a') as f: f.write(hdr)

tf = CFG['tf_start']
dl = da = de = vl = va = ve = rc = 0.0

for step in range(1, CFG['max_steps']+1):
    tf = max(CFG['tf_min'], tf * CFG['tf_decay'])
    ids, starts = make_batch(CFG['batch'], tok)

    dual.train()
    d_opt.zero_grad()
    d_logits = dual(ids, tf=tf)
    d_loss, d_acc = loss_and_acc(d_logits, ids, starts, tok.pad_id)
    d_loss.backward()
    torch.nn.utils.clip_grad_norm_(dual.parameters(), 5.0)
    d_opt.step()

    vanilla.train()
    v_opt.zero_grad()
    v_logits = vanilla(ids, tf=tf)
    v_loss, v_acc = loss_and_acc(v_logits, ids, starts, tok.pad_id)
    v_loss.backward()
    torch.nn.utils.clip_grad_norm_(vanilla.parameters(), 5.0)
    v_opt.step()

    dl += d_loss.item(); da += d_acc; de += mean_entropy(d_logits)
    vl += v_loss.item(); va += v_acc; ve += mean_entropy(v_logits)
    rc += 1

    if step % CFG['log_every'] == 0:
        la = dual.left.anchor.item()
        ra = dual.right.anchor.item()
        msg = (f"step={step:6d} | "
               f"DUAL loss={dl/rc:.4f} acc={da/rc*100:.1f}% ent={de/rc:.3f} "
               f"La={la:.4f} Ra={ra:.4f} | "
               f"VANILLA loss={vl/rc:.4f} acc={va/rc*100:.1f}% ent={ve/rc:.3f} | "
               f"tf={tf:.3f}")
        print(msg)
        with open(LOG_PATH,'a') as f: f.write(msg+'\n')
        sys.stdout.flush()
        dl = da = de = vl = va = ve = rc = 0.0

    if step % CFG['probe_every'] == 0:
        run_probe(dual, vanilla, tok, step)

print(f"\nDONE -- step={step}")
print(f"Left anchor:  {dual.left.anchor.item():.6f}")
print(f"Right anchor: {dual.right.anchor.item():.6f}")
print(f"phi=4/pi:     {PHI_CL:.6f}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

  DPPU-VRU v22  |  DUAL HEMISPHERE + AGREEMENT GATE
  device=cuda
  DUAL params=200,511  |  Vanilla params=46,849

  Left hemisphere  -- independent DPPU stream
  Right hemisphere -- independent DPPU stream
  Agreement gate   -- fires when both streams align
  Focus layer      -- arbitrates, outputs when agreed
  Probe: dual | left-only | right-only | vanilla

  Format: ones->carry->tens->ans (v21 proven)
  Both anchors free -- no regularization



KeyboardInterrupt: 

In [ ]:
# ============================================================
# DPPU-VRU v22 -- DUAL HEMISPHERE + AGREEMENT GATE
# Dylan Michael Scott -- Horizon Tech
#
# Two independent DPPU streams (hemispheres)
# Agreement gate fires when both streams align
# Focus layer arbitrates the output
# Per-hemisphere probe so we can see what each side is doing
#
# Vectorized cell and loss -- no Python loops, runs fast
# Format: ones->carry->tens->ans (v21 proven)
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import torch, torch.nn as nn, torch.nn.functional as F
import math, random, os, sys
from datetime import datetime

DRIVE_DIR = '/content/drive/MyDrive/dppu_vru'
LOG_PATH  = os.path.join(DRIVE_DIR, 'v22_log.txt')
os.makedirs(DRIVE_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

PI_CL  = math.pi
PHI_CL = 4.0 / math.pi
D_CAP  = 4
EPS    = 1e-7

def pi_dyn(d):  return 4.0 - (4.0 - PI_CL) * torch.exp(-d)
def phi_dyn(d): return PHI_CL * torch.exp(-d) + (1.0 - torch.exp(-d))
def omega(d):   return (pi_dyn(d) * phi_dyn(d)) / (1.0 + d + EPS)

CFG = dict(
    hidden      = 130,
    lr          = 1e-3,
    batch       = 32,
    max_steps   = 20_000,
    log_every   = 200,
    probe_every = 2_000,
    tf_start    = 0.9,
    tf_min      = 0.05,
    tf_decay    = 0.9995,
)

class MathTokenizer:
    SPECIAL = ['<pad>', '<bos>', '<eos>', '<unk>']
    def __init__(self):
        self.vocab = {s: i for i, s in enumerate(self.SPECIAL)}
        for c in '0123456789+-*/=:,abcdefghijklmnopqrstuvwxyz_ ':
            if c not in self.vocab:
                self.vocab[c] = len(self.vocab)
        self.inv    = {v: k for k, v in self.vocab.items()}
        self.pad_id = self.vocab['<pad>']
        self.bos_id = self.vocab['<bos>']
        self.eos_id = self.vocab['<eos>']
        self.eq_id  = self.vocab['=']
    @property
    def vocab_size(self): return len(self.vocab)
    def encode(self, text, add_bos=True, add_eos=True):
        return (([self.bos_id] if add_bos else []) +
                [self.vocab.get(c, self.vocab['<unk>']) for c in text] +
                ([self.eos_id] if add_eos else []))
    def decode(self, ids):
        return ''.join(self.inv.get(i,'?') for i in ids
                       if self.inv.get(i,'?') not in self.SPECIAL)

def make_step_example(a, b):
    ones     = (a % 10) + (b % 10)
    carry    = ones // 10
    tens_sum = (a // 10) + (b // 10) + carry
    return f"{a}+{b}=ones:{ones},carry:{carry},tens:{tens_sum},ans:{a+b}"

def gen_add2():
    a, b = random.randint(10,99), random.randint(10,99)
    return make_step_example(a, b)

def make_batch(bsz, tok):
    seqs, starts = [], []
    for _ in range(bsz):
        raw   = gen_add2()
        ids   = tok.encode(raw)
        start = next((j+1 for j,t in enumerate(ids) if t==tok.eq_id), 1)
        seqs.append(ids); starts.append(start)
    ml  = max(len(s) for s in seqs)
    pad = [s + [tok.pad_id]*(ml-len(s)) for s in seqs]
    return torch.tensor(pad, dtype=torch.long, device=device), starts

class DPPUHemisphere(nn.Module):
    def __init__(self, in_dim, hid):
        super().__init__()
        self.n   = D_CAP + 1
        self.ds  = hid // self.n
        self.hid = hid
        self.Wx  = nn.Linear(in_dim, hid)
        self.Wh  = nn.Linear(hid, hid, bias=False)
        self.Wc  = nn.Linear(hid, self.n, bias=False)
        self.Wo  = nn.Linear(hid, hid, bias=False)
        self.anchor = nn.Parameter(torch.tensor(PHI_CL))

    def forward(self, x, h, C):
        B  = x.size(0)
        xp = self.Wx(x)
        wh = self.Wh(h)
        h_  = h.view(B, self.n, self.ds)
        xp_ = xp.view(B, self.n, self.ds)
        wh_ = wh.view(B, self.n, self.ds)
        di     = torch.tanh(torch.abs(h_) / (h_.std(dim=-1, keepdim=True) + EPS))
        h_rec  = (phi_dyn(di) / PHI_CL) * wh_
        x_rec  = (pi_dyn(di)  / PI_CL)  * xp_
        gate   = torch.sigmoid(omega(di))
        anchor = C.unsqueeze(-1) * self.anchor
        h_new  = torch.tanh(h_rec + x_rec + anchor) * gate
        h2 = self.Wo(h_new.view(B, self.hid))
        C2 = torch.clamp(torch.tanh(0.1*C + 0.01*self.Wc(h2)), -0.5, 0.5)
        return h2, C2

class DualHemisphereModel(nn.Module):
    def __init__(self, vsz, hid=130):
        super().__init__()
        self.hid   = hid
        self.n     = D_CAP + 1
        self.emb   = nn.Embedding(vsz, hid)
        self.left  = DPPUHemisphere(hid, hid)
        self.right = DPPUHemisphere(hid, hid)
        self.agree = nn.Linear(hid * 2, hid)
        self.out   = nn.Linear(hid, vsz)

    def zeros(self, B):
        h = torch.zeros(B, self.hid, device=device)
        C = torch.full((B, self.n), PHI_CL, device=device)
        return h, C

    def forward(self, ids, tf=1.0):
        B, T = ids.shape
        lh, lC = self.zeros(B)
        rh, rC = self.zeros(B)
        outs = []
        for t in range(T-1):
            tok = ids[:,t] if (t==0 or random.random()<tf) else outs[-1].argmax(-1)
            x      = self.emb(tok)
            lh, lC = self.left(x,  lh.detach(), lC.detach())
            rh, rC = self.right(x, rh.detach(), rC.detach())
            # agreement: simple average weighted by gate
            gate   = torch.sigmoid(self.agree(torch.cat([lh, rh], dim=-1)))
            merged = gate * lh + (1 - gate) * rh
            outs.append(self.out(merged))
        return torch.stack(outs, dim=1)

class VanillaModel(nn.Module):
    def __init__(self, vsz, hid=130):
        super().__init__()
        self.hid  = hid
        self.emb  = nn.Embedding(vsz, hid)
        self.cell = nn.RNNCell(hid, hid)
        self.out  = nn.Linear(hid, vsz)

    def forward(self, ids, tf=1.0):
        B, T = ids.shape
        h    = torch.zeros(B, self.hid, device=device)
        outs = []
        for t in range(T-1):
            tok  = ids[:,t] if (t==0 or random.random()<tf) else outs[-1].argmax(-1)
            x    = self.emb(tok)
            h    = self.cell(x, h)
            outs.append(self.out(h))
        return torch.stack(outs, dim=1)

def loss_and_acc(logits, ids, starts, pad_id):
    B, T, V = logits.shape
    targets = ids[:, 1:]
    mask    = (targets != pad_id)
    for b in range(B):
        if starts[b]-1 > 0:
            mask[b, :starts[b]-1] = False
    fl = logits.reshape(-1, V)
    ft = targets.reshape(-1)
    fm = mask.reshape(-1)
    if fm.sum() == 0:
        return torch.tensor(0.0, device=logits.device), 0.0
    loss = F.cross_entropy(fl[fm], ft[fm])
    acc  = (fl[fm].argmax(-1) == ft[fm]).float().mean().item()
    return loss, acc

def mean_entropy(logits):
    p = torch.softmax(logits, dim=-1)
    return -(p * torch.log(p + EPS)).sum(-1).mean().item()

@torch.no_grad()
def gen_dual(model, tok, prompt_ids, max_len=30):
    inp    = torch.tensor([prompt_ids], dtype=torch.long, device=device)
    lh, lC = model.zeros(1)
    rh, rC = model.zeros(1)
    for t in range(inp.size(1)-1):
        x      = model.emb(inp[:,t])
        lh, lC = model.left(x,  lh, lC)
        rh, rC = model.right(x, rh, rC)
    cur, out = inp[:,-1], []
    for _ in range(max_len):
        x      = model.emb(cur)
        lh, lC = model.left(x,  lh.detach(), lC.detach())
        rh, rC = model.right(x, rh.detach(), rC.detach())
        gate   = torch.sigmoid(model.agree(torch.cat([lh, rh], dim=-1)))
        merged = gate * lh + (1 - gate) * rh
        lg     = model.out(merged)
        nxt    = lg.argmax(-1)
        char   = tok.inv.get(nxt.item(), '?')
        if char in ('<eos>','<pad>'): break
        out.append(char); cur = nxt
    return ''.join(out)

@torch.no_grad()
def gen_hemi(model, tok, prompt_ids, side='left', max_len=30):
    inp    = torch.tensor([prompt_ids], dtype=torch.long, device=device)
    cell   = model.left if side == 'left' else model.right
    h, C   = model.zeros(1)
    for t in range(inp.size(1)-1):
        x    = model.emb(inp[:,t])
        h, C = cell(x, h, C)
    cur, out = inp[:,-1], []
    for _ in range(max_len):
        x    = model.emb(cur)
        h, C = cell(x, h, C)
        lg   = model.out(h)
        nxt  = lg.argmax(-1)
        char = tok.inv.get(nxt.item(), '?')
        if char in ('<eos>','<pad>'): break
        out.append(char); cur = nxt
    return ''.join(out)

@torch.no_grad()
def gen_vanilla(model, tok, prompt_ids, max_len=30):
    inp = torch.tensor([prompt_ids], dtype=torch.long, device=device)
    h   = torch.zeros(1, model.hid, device=device)
    for t in range(inp.size(1)-1):
        x = model.emb(inp[:,t])
        h = model.cell(x, h)
    cur, out = inp[:,-1], []
    for _ in range(max_len):
        x    = model.emb(cur)
        h    = model.cell(x, h)
        lg   = model.out(h)
        nxt  = lg.argmax(-1)
        char = tok.inv.get(nxt.item(), '?')
        if char in ('<eos>','<pad>'): break
        out.append(char); cur = nxt
    return ''.join(out)

def extract_answer(output):
    if 'ans:' in output:
        try: return output.split('ans:')[-1].split(',')[0].strip()
        except: pass
    return output

PROBE_EX = [(12,34,46),(55,27,82),(73,18,91),(99,11,110),(64,36,100)]

@torch.no_grad()
def run_probe(dual, vanilla, tok, step):
    dual.eval(); vanilla.eval()
    d_ok = v_ok = 0
    la  = dual.left.anchor.item()
    ra  = dual.right.anchor.item()
    agw = dual.agree.weight.abs().mean().item()
    lines = [
        f"\n  PROBE step={step}",
        f"  L-anchor={la:.6f}  R-anchor={ra:.6f}  phi={PHI_CL:.6f}",
        f"  agree_gate={agw:.4f}  focus={fow:.4f}",
        f"  {'PROB':<10} {'ANS':<5} {'DUAL OUTPUT':<26} {'LEFT':<22} {'RIGHT':<22} {'VANILLA':<18}",
        f"  {'-'*100}",
    ]
    for a, b, ans in PROBE_EX:
        prompt = f"{a}+{b}="
        ids    = tok.encode(prompt, add_bos=True, add_eos=False)
        dp     = gen_dual(dual, tok, ids)
        lo     = gen_hemi(dual, tok, ids, 'left')
        ro     = gen_hemi(dual, tok, ids, 'right')
        vp     = gen_vanilla(vanilla, tok, ids)
        dp_ans = extract_answer(dp)
        dh = '✓' if dp_ans == str(ans) else '✗'
        vh = '✓' if extract_answer(vp) == str(ans) else '✗'
        if dp_ans == str(ans): d_ok += 1
        if extract_answer(vp) == str(ans): v_ok += 1
        lines.append(f"  {prompt:<10} {str(ans):<5} {dh} {dp:<24} {lo:<22} {ro:<22} {vh} {vp}")
    lines.append(f"\n  DUAL: {d_ok}/5   VANILLA: {v_ok}/5\n")
    msg = '\n'.join(lines)
    print(msg)
    with open(LOG_PATH,'a') as f: f.write(msg+'\n')
    sys.stdout.flush()
    dual.train(); vanilla.train()

# ============================================================
# MAIN
# ============================================================
tok     = MathTokenizer()
dual    = DualHemisphereModel(tok.vocab_size, CFG['hidden']).to(device)
vanilla = VanillaModel(tok.vocab_size, CFG['hidden']).to(device)
d_opt   = torch.optim.Adam(dual.parameters(),    lr=CFG['lr'])
v_opt   = torch.optim.Adam(vanilla.parameters(), lr=CFG['lr'])

d_params = sum(p.numel() for p in dual.parameters())
v_params = sum(p.numel() for p in vanilla.parameters())

hdr = (f"\n{'='*62}\n"
       f"  DPPU-VRU v22  |  DUAL HEMISPHERE + AGREEMENT GATE\n"
       f"  device={device}\n"
       f"  DUAL params={d_params:,}  |  Vanilla params={v_params:,}\n"
       f"\n  Left hemisphere  -- independent DPPU stream\n"
       f"  Right hemisphere -- independent DPPU stream\n"
       f"  Agreement gate   -- fires when both streams align\n"
       f"  Focus layer      -- arbitrates, outputs when agreed\n"
       f"  Probe: dual | left-only | right-only | vanilla\n"
       f"\n  Format: ones->carry->tens->ans (v21 proven)\n"
       f"  Both anchors free -- no regularization\n"
       f"{'='*62}\n")
print(hdr)
with open(LOG_PATH,'a') as f: f.write(hdr)

tf = CFG['tf_start']
dl = da = de = vl = va = ve = rc = 0.0

for step in range(1, CFG['max_steps']+1):
    tf = max(CFG['tf_min'], tf * CFG['tf_decay'])
    ids, starts = make_batch(CFG['batch'], tok)

    dual.train()
    d_opt.zero_grad()
    d_logits = dual(ids, tf=tf)
    d_loss, d_acc = loss_and_acc(d_logits, ids, starts, tok.pad_id)
    d_loss.backward()
    torch.nn.utils.clip_grad_norm_(dual.parameters(), 5.0)
    d_opt.step()

    vanilla.train()
    v_opt.zero_grad()
    v_logits = vanilla(ids, tf=tf)
    v_loss, v_acc = loss_and_acc(v_logits, ids, starts, tok.pad_id)
    v_loss.backward()
    torch.nn.utils.clip_grad_norm_(vanilla.parameters(), 5.0)
    v_opt.step()

    dl += d_loss.item(); da += d_acc; de += mean_entropy(d_logits)
    vl += v_loss.item(); va += v_acc; ve += mean_entropy(v_logits)
    rc += 1

    if step % CFG['log_every'] == 0:
        la = dual.left.anchor.item()
        ra = dual.right.anchor.item()
        msg = (f"step={step:6d} | "
               f"DUAL loss={dl/rc:.4f} acc={da/rc*100:.1f}% ent={de/rc:.3f} "
               f"La={la:.4f} Ra={ra:.4f} | "
               f"VANILLA loss={vl/rc:.4f} acc={va/rc*100:.1f}% ent={ve/rc:.3f} | "
               f"tf={tf:.3f}")
        print(msg)
        with open(LOG_PATH,'a') as f: f.write(msg+'\n')
        sys.stdout.flush()
        dl = da = de = vl = va = ve = rc = 0.0

    if step % CFG['probe_every'] == 0:
        run_probe(dual, vanilla, tok, step)

print(f"\nDONE -- step={step}")
print(f"Left anchor:  {dual.left.anchor.item():.6f}")
print(f"Right anchor: {dual.right.anchor.item():.6f}")
print(f"phi=4/pi:     {PHI_CL:.6f}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

  DPPU-VRU v22  |  DUAL HEMISPHERE + AGREEMENT GATE
  device=cuda
  DUAL params=149,681  |  Vanilla params=46,849

  Left hemisphere  -- independent DPPU stream
  Right hemisphere -- independent DPPU stream
  Agreement gate   -- fires when both streams align
  Focus layer      -- arbitrates, outputs when agreed
  Probe: dual | left-only | right-only | vanilla

  Format: ones->carry->tens->ans (v21 proven)
  Both anchors free -- no regularization



KeyboardInterrupt: 

In [ ]:
# ============================================================
# DPPU-VRU v21 -- TEACH THE PROCESS NOT THE ANSWER
# Dylan Michael Scott -- Horizon Tech
#
# Instead of: 73+18=91
# We give it:  73+18=ones:11,carry:1,tens:9,ans:91
#
# The model follows the actual computation steps.
# Ones column first. Carry. Tens column. Then answer.
# This mirrors how arithmetic is actually performed --
# not as a lookup, but as a sequential process.
#
# DPPU vs Vanilla side by side. Same batches. Honest test.
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import torch, torch.nn as nn, torch.nn.functional as F
import math, random, os, json, sys
from datetime import datetime

DRIVE_DIR = '/content/drive/MyDrive/dppu_vru'
LOG_PATH  = os.path.join(DRIVE_DIR, 'v21_log.txt')
os.makedirs(DRIVE_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

PI_CL  = math.pi
PHI_CL = 4.0 / math.pi
D_CAP  = 4
EPS    = 1e-7

def pi_dyn(d):  return 4.0 - (4.0 - PI_CL) * torch.exp(-d)
def phi_dyn(d): return PHI_CL * torch.exp(-d) + (1.0 - torch.exp(-d))
def omega(d):   return (pi_dyn(d) * phi_dyn(d)) / (1.0 + d + EPS)

def compute_delta(h):
    return torch.tanh(torch.abs(h) / (h.std(dim=-1, keepdim=True) + EPS))

CFG = dict(
    hidden       = 260,
    lr           = 1e-3,
    batch        = 32,
    max_steps    = 20_000,
    log_every    = 200,
    probe_every  = 2_000,
    tf_start     = 0.9,
    tf_min       = 0.05,
    tf_decay     = 0.9995,
)

# ============================================================
# TOKENIZER -- extended for step tokens
# ============================================================
class MathTokenizer:
    SPECIAL = ['<pad>', '<bos>', '<eos>', '<unk>']

    def __init__(self):
        self.vocab = {s: i for i, s in enumerate(self.SPECIAL)}
        # digits, operators, separators
        for c in '0123456789+-*/=:,abcdefghijklmnopqrstuvwxyz_ ':
            if c not in self.vocab:
                self.vocab[c] = len(self.vocab)
        self.inv = {v: k for k, v in self.vocab.items()}
        self.pad_id = self.vocab['<pad>']
        self.bos_id = self.vocab['<bos>']
        self.eos_id = self.vocab['<eos>']
        self.eq_id  = self.vocab['=']

    @property
    def vocab_size(self): return len(self.vocab)

    def encode(self, text, add_bos=True, add_eos=True):
        return (([self.bos_id] if add_bos else []) +
                [self.vocab.get(c, self.vocab['<unk>']) for c in text] +
                ([self.eos_id] if add_eos else []))

    def decode(self, ids):
        return ''.join(self.inv.get(i,'?') for i in ids
                       if self.inv.get(i,'?') not in self.SPECIAL)

# ============================================================
# DATA -- step-by-step format
#
# 73+18= -> ones:11,carry:1,tens:9,ans:91
#
# The answer is still at the end (ans:XX)
# but the model has to work through the process to get there
# ============================================================
def make_step_example(a, b):
    ones      = (a % 10) + (b % 10)
    carry     = ones // 10
    tens_sum  = (a // 10) + (b // 10) + carry
    answer    = a + b
    # format: question=ones:N,carry:N,tens:N,ans:N
    prompt  = f"{a}+{b}="
    process = f"ones:{ones},carry:{carry},tens:{tens_sum},ans:{answer}"
    return prompt + process

def gen_add2():
    a, b = random.randint(10,99), random.randint(10,99)
    return make_step_example(a, b)

def make_batch(bsz, tok):
    seqs, starts = [], []
    for _ in range(bsz):
        raw   = gen_add2()
        ids   = tok.encode(raw)
        # answer starts after '=' -- model predicts the process
        start = next((j+1 for j,t in enumerate(ids) if t==tok.eq_id), 1)
        seqs.append(ids); starts.append(start)
    ml  = max(len(s) for s in seqs)
    pad = [s + [tok.pad_id]*(ml-len(s)) for s in seqs]
    return torch.tensor(pad, dtype=torch.long, device=device), starts

# ============================================================
# DPPU MODEL
# ============================================================
class DPPUCell(nn.Module):
    def __init__(self, in_dim, hid):
        super().__init__()
        self.n  = D_CAP + 1
        self.ds = hid // self.n
        self.Wx = nn.Linear(in_dim, hid)
        self.Wh = nn.ModuleList([nn.Linear(hid, self.ds, bias=False) for _ in range(self.n)])
        self.Wc = nn.Linear(hid, self.n, bias=False)
        self.Wo = nn.Linear(hid, hid, bias=False)
        self.anchor = nn.Parameter(torch.tensor(PHI_CL))

    def forward(self, x, h, C):
        xp, parts = self.Wx(x), []
        for i in range(self.n):
            s, e   = i*self.ds, (i+1)*self.ds
            di     = compute_delta(h[:, s:e])
            h_rec  = (phi_dyn(di) / PHI_CL) * self.Wh[i](h)
            x_rec  = (pi_dyn(di)  / PI_CL)  * xp[:, s:e]
            gate   = torch.sigmoid(omega(di))
            anchor = C[:, i:i+1] * self.anchor
            parts.append(torch.tanh(h_rec + x_rec + anchor) * gate)
        h2 = self.Wo(torch.cat(parts, dim=-1))
        C2 = torch.clamp(torch.tanh(0.1*C + 0.01*self.Wc(h2)), -0.5, 0.5)
        return h2, C2


class DPPUModel(nn.Module):
    def __init__(self, vsz, hid=260):
        super().__init__()
        self.hid  = hid
        self.n    = D_CAP + 1
        self.emb  = nn.Embedding(vsz, hid)
        self.cell = DPPUCell(hid, hid)
        self.out  = nn.Linear(hid, vsz)

    def zeros(self, B):
        h = torch.zeros(B, self.hid, device=device)
        C = torch.full((B, self.n), PHI_CL, device=device)
        return h, C

    def forward(self, ids, tf=1.0):
        B, T = ids.shape
        h, C = self.zeros(B)
        outs  = []
        for t in range(T-1):
            tok  = ids[:,t] if (t==0 or random.random()<tf) else outs[-1].argmax(-1)
            x    = self.emb(tok)
            h, C = self.cell(x, h, C)
            outs.append(self.out(h))
        return torch.stack(outs, dim=1)

# ============================================================
# VANILLA MODEL
# ============================================================
class VanillaModel(nn.Module):
    def __init__(self, vsz, hid=260):
        super().__init__()
        self.hid  = hid
        self.emb  = nn.Embedding(vsz, hid)
        self.cell = nn.RNNCell(hid, hid)
        self.out  = nn.Linear(hid, vsz)

    def forward(self, ids, tf=1.0):
        B, T = ids.shape
        h    = torch.zeros(B, self.hid, device=device)
        outs = []
        for t in range(T-1):
            tok  = ids[:,t] if (t==0 or random.random()<tf) else outs[-1].argmax(-1)
            x    = self.emb(tok)
            h    = self.cell(x, h)
            outs.append(self.out(h))
        return torch.stack(outs, dim=1)

# ============================================================
# LOSS
# ============================================================
def loss_and_acc(logits, ids, starts, pad_id):
    B = logits.size(0)
    L = torch.tensor(0.0, device=logits.device)
    ok = tot = 0
    for b in range(B):
        for t in range(starts[b]-1, ids.size(1)-1):
            tgt = ids[b,t+1].item()
            if tgt == pad_id: break
            L += F.cross_entropy(logits[b,t].unsqueeze(0),
                                 torch.tensor([tgt], device=logits.device))
            if logits[b,t].argmax(-1).item() == tgt: ok += 1
            tot += 1
    return (L/tot, ok/tot) if tot>0 else (L, 0.0)

# ============================================================
# GENERATION
# ============================================================
@torch.no_grad()
def gen_dppu(model, tok, prompt_ids, max_len=30):
    inp     = torch.tensor([prompt_ids], dtype=torch.long, device=device)
    h, C    = model.zeros(1)
    for t in range(inp.size(1)-1):
        x    = model.emb(inp[:,t])
        h, C = model.cell(x, h, C)
    cur, out = inp[:,-1], []
    for _ in range(max_len):
        x    = model.emb(cur)
        h, C = model.cell(x, h, C)
        lg   = model.out(h)
        nxt  = lg.argmax(-1)
        char = tok.inv.get(nxt.item(), '?')
        if char in ('<eos>','<pad>'): break
        out.append(char); cur = nxt
    return ''.join(out)

@torch.no_grad()
def gen_vanilla(model, tok, prompt_ids, max_len=30):
    inp = torch.tensor([prompt_ids], dtype=torch.long, device=device)
    h   = torch.zeros(1, model.hid, device=device)
    for t in range(inp.size(1)-1):
        x = model.emb(inp[:,t])
        h = model.cell(x, h)
    cur, out = inp[:,-1], []
    for _ in range(max_len):
        x  = model.emb(cur)
        h  = model.cell(x, h)
        lg = model.out(h)
        nxt  = lg.argmax(-1)
        char = tok.inv.get(nxt.item(), '?')
        if char in ('<eos>','<pad>'): break
        out.append(char); cur = nxt
    return ''.join(out)

def extract_answer(output):
    # pull the number after 'ans:' if present
    if 'ans:' in output:
        try: return output.split('ans:')[-1].split(',')[0].strip()
        except: pass
    return output

# ============================================================
# PROBE
# ============================================================
PROBE_EX = [
    (12, 34, 46),
    (55, 27, 82),
    (73, 18, 91),
    (99, 11, 110),
    (64, 36, 100),
]

@torch.no_grad()
def run_probe(dppu, vanilla, tok, step):
    dppu.eval(); vanilla.eval()
    d_ok = v_ok = 0
    lines = [f"\n  PROBE step={step}  anchor={dppu.cell.anchor.item():.6f}"]
    lines.append(f"  {'PROBLEM':<12} {'TARGET':<6} {'DPPU OUTPUT':<28} {'VANILLA OUTPUT':<28} {'ANS'}")
    lines.append(f"  {'-'*85}")
    for a, b, ans in PROBE_EX:
        prompt  = f"{a}+{b}="
        target  = make_step_example(a, b).split('=')[1]
        ids     = tok.encode(prompt, add_bos=True, add_eos=False)
        dp_out  = gen_dppu(dppu, tok, ids)
        vp_out  = gen_vanilla(vanilla, tok, ids)
        dp_ans  = extract_answer(dp_out)
        vp_ans  = extract_answer(vp_out)
        dh = '✓' if dp_ans == str(ans) else '✗'
        vh = '✓' if vp_ans == str(ans) else '✗'
        if dp_ans == str(ans): d_ok += 1
        if vp_ans == str(ans): v_ok += 1
        lines.append(f"  {prompt:<12} {str(ans):<6} {dh} {dp_out:<26} {vh} {vp_out:<26}")
    lines.append(f"\n  DPPU: {d_ok}/5 correct answers    VANILLA: {v_ok}/5 correct answers\n")
    msg = '\n'.join(lines)
    print(msg)
    with open(LOG_PATH,'a') as f: f.write(msg+'\n')
    sys.stdout.flush()
    dppu.train(); vanilla.train()

# ============================================================
# MAIN
# ============================================================
tok     = MathTokenizer()
dppu    = DPPUModel(tok.vocab_size, CFG['hidden']).to(device)
vanilla = VanillaModel(tok.vocab_size, CFG['hidden']).to(device)
d_opt   = torch.optim.Adam(dppu.parameters(),    lr=CFG['lr'])
v_opt   = torch.optim.Adam(vanilla.parameters(), lr=CFG['lr'])

d_params = sum(p.numel() for p in dppu.parameters())
v_params = sum(p.numel() for p in vanilla.parameters())

# print a sample to show the format
sample = gen_add2()
hdr = (f"\n{'='*62}\n"
       f"  DPPU-VRU v21  |  TEACH THE PROCESS\n"
       f"  device={device}\n"
       f"  DPPU params={d_params:,}  |  Vanilla params={v_params:,}\n"
       f"\n  Example training sequence:\n"
       f"  INPUT:  {sample.split('=')[0]}=\n"
       f"  TARGET: {sample.split('=')[1]}\n"
       f"\n  Model must predict: ones -> carry -> tens -> answer\n"
       f"  Not just the answer. The whole process.\n"
       f"{'='*62}\n")
print(hdr)
with open(LOG_PATH,'a') as f: f.write(hdr)

tf = CFG['tf_start']
dl = da = vl = va = rc = 0.0

for step in range(1, CFG['max_steps']+1):
    tf = max(CFG['tf_min'], tf * CFG['tf_decay'])
    ids, starts = make_batch(CFG['batch'], tok)

    # DPPU
    dppu.train()
    d_opt.zero_grad()
    d_logits = dppu(ids, tf=tf)
    d_loss, d_acc = loss_and_acc(d_logits, ids, starts, tok.pad_id)
    d_loss.backward()
    torch.nn.utils.clip_grad_norm_(dppu.parameters(), 5.0)
    d_opt.step()

    # Vanilla
    vanilla.train()
    v_opt.zero_grad()
    v_logits = vanilla(ids, tf=tf)
    v_loss, v_acc = loss_and_acc(v_logits, ids, starts, tok.pad_id)
    v_loss.backward()
    torch.nn.utils.clip_grad_norm_(vanilla.parameters(), 5.0)
    v_opt.step()

    dl += d_loss.item(); da += d_acc
    vl += v_loss.item(); va += v_acc
    rc += 1

    if step % CFG['log_every'] == 0:
        msg = (f"step={step:6d} | "
               f"DPPU  loss={dl/rc:.4f} acc={da/rc*100:.1f}%  anchor={dppu.cell.anchor.item():.4f} | "
               f"VANILLA  loss={vl/rc:.4f} acc={va/rc*100:.1f}% | "
               f"tf={tf:.3f}")
        print(msg)
        with open(LOG_PATH,'a') as f: f.write(msg+'\n')
        sys.stdout.flush()
        dl = da = vl = va = rc = 0.0

    if step % CFG['probe_every'] == 0:
        run_probe(dppu, vanilla, tok, step)

print(f"\nDONE -- step={step}")
print(f"Final anchor: {dppu.cell.anchor.item():.6f}  (started at {PHI_CL:.6f})")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

  DPPU-VRU v21  |  TEACH THE PROCESS
  device=cuda
  DPPU params=229,890  |  Vanilla params=161,249

  Example training sequence:
  INPUT:  98+64=
  TARGET: ones:12,carry:1,tens:16,ans:162

  Model must predict: ones -> carry -> tens -> answer
  Not just the answer. The whole process.



KeyboardInterrupt: 

In [2]:
# ============================================================
# DPPU-VRU v21 -- TEACH THE PROCESS NOT THE ANSWER
# Dylan Michael Scott -- Horizon Tech
#
# Instead of: 73+18=91
# We give it:  73+18=ones:11,carry:1,tens:9,ans:91
#
# The model follows the actual computation steps.
# Ones column first. Carry. Tens column. Then answer.
# This mirrors how arithmetic is actually performed --
# not as a lookup, but as a sequential process.
#
# DPPU vs Vanilla side by side. Same batches. Honest test.
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import torch, torch.nn as nn, torch.nn.functional as F
import math, random, os, json, sys
from datetime import datetime

DRIVE_DIR = '/content/drive/MyDrive/dppu_vru'
LOG_PATH  = os.path.join(DRIVE_DIR, 'v21_log.txt')
os.makedirs(DRIVE_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

PI_CL  = math.pi
PHI_CL = 4.0 / math.pi
D_CAP  = 4
EPS    = 1e-7

def pi_dyn(d):  return 4.0 - (4.0 - PI_CL) * torch.exp(-d)
def phi_dyn(d): return PHI_CL * torch.exp(-d) + (1.0 - torch.exp(-d))
def omega(d):   return (pi_dyn(d) * phi_dyn(d)) / (1.0 + d + EPS)

def compute_delta(h):
    return torch.tanh(torch.abs(h) / (h.std(dim=-1, keepdim=True) + EPS))

CFG = dict(
    hidden       = 260,
    lr           = 1e-3,
    batch        = 32,
    max_steps    = 20_000,
    log_every    = 200,
    probe_every  = 2_000,
    tf_start     = 0.9,
    tf_min       = 0.05,
    tf_decay     = 0.9995,
)

# ============================================================
# TOKENIZER -- extended for step tokens
# ============================================================
class MathTokenizer:
    SPECIAL = ['<pad>', '<bos>', '<eos>', '<unk>']

    def __init__(self):
        self.vocab = {s: i for i, s in enumerate(self.SPECIAL)}
        # digits, operators, separators
        for c in '0123456789+-*/=:,abcdefghijklmnopqrstuvwxyz_ ':
            if c not in self.vocab:
                self.vocab[c] = len(self.vocab)
        self.inv = {v: k for k, v in self.vocab.items()}
        self.pad_id = self.vocab['<pad>']
        self.bos_id = self.vocab['<bos>']
        self.eos_id = self.vocab['<eos>']
        self.eq_id  = self.vocab['=']

    @property
    def vocab_size(self): return len(self.vocab)

    def encode(self, text, add_bos=True, add_eos=True):
        return (([self.bos_id] if add_bos else []) +
                [self.vocab.get(c, self.vocab['<unk>']) for c in text] +
                ([self.eos_id] if add_eos else []))

    def decode(self, ids):
        return ''.join(self.inv.get(i,'?') for i in ids
                       if self.inv.get(i,'?') not in self.SPECIAL)

# ============================================================
# DATA -- step-by-step format
#
# 73+18= -> ones:11,carry:1,tens:9,ans:91
#
# The answer is still at the end (ans:XX)
# but the model has to work through the process to get there
# ============================================================
def make_step_example(a, b):
    ones      = (a % 10) + (b % 10)
    carry     = ones // 10
    tens_sum  = (a // 10) + (b // 10) + carry
    answer    = a + b
    # format: question=ones:N,carry:N,tens:N,ans:N
    prompt  = f"{a}+{b}="
    process = f"ones:{ones},carry:{carry},tens:{tens_sum},ans:{answer}"
    return prompt + process

def gen_add2():
    a, b = random.randint(10,99), random.randint(10,99)
    return make_step_example(a, b)

def make_batch(bsz, tok):
    seqs, starts = [], []
    for _ in range(bsz):
        raw   = gen_add2()
        ids   = tok.encode(raw)
        # answer starts after '=' -- model predicts the process
        start = next((j+1 for j,t in enumerate(ids) if t==tok.eq_id), 1)
        seqs.append(ids); starts.append(start)
    ml  = max(len(s) for s in seqs)
    pad = [s + [tok.pad_id]*(ml-len(s)) for s in seqs]
    return torch.tensor(pad, dtype=torch.long, device=device), starts

# ============================================================
# DPPU MODEL
# ============================================================
class DPPUCell(nn.Module):
    def __init__(self, in_dim, hid):
        super().__init__()
        self.n   = D_CAP + 1
        self.ds  = hid // self.n
        self.hid = hid
        self.Wx  = nn.Linear(in_dim, hid)
        self.Wh  = nn.Linear(hid, hid, bias=False)
        self.Wc  = nn.Linear(hid, self.n, bias=False)
        self.Wo  = nn.Linear(hid, hid, bias=False)
        self.anchor = nn.Parameter(torch.tensor(PHI_CL))

    def forward(self, x, h, C):
        B   = x.size(0)
        xp  = self.Wx(x)
        wh  = self.Wh(h)
        h_  = h.view(B, self.n, self.ds)
        xp_ = xp.view(B, self.n, self.ds)
        wh_ = wh.view(B, self.n, self.ds)
        di     = torch.tanh(torch.abs(h_) / (h_.std(dim=-1, keepdim=True) + EPS))
        h_rec  = (phi_dyn(di) / PHI_CL) * wh_
        x_rec  = (pi_dyn(di)  / PI_CL)  * xp_
        gate   = torch.sigmoid(omega(di))
        anchor = C.unsqueeze(-1) * self.anchor
        h_new  = torch.tanh(h_rec + x_rec + anchor) * gate
        h2 = self.Wo(h_new.view(B, self.hid))
        C2 = torch.clamp(torch.tanh(0.1*C + 0.01*self.Wc(h2)), -0.5, 0.5)
        return h2, C2


class DPPUModel(nn.Module):
    def __init__(self, vsz, hid=260):
        super().__init__()
        self.hid  = hid
        self.n    = D_CAP + 1
        self.emb  = nn.Embedding(vsz, hid)
        self.cell = DPPUCell(hid, hid)
        self.out  = nn.Linear(hid, vsz)

    def zeros(self, B):
        h = torch.zeros(B, self.hid, device=device)
        C = torch.full((B, self.n), PHI_CL, device=device)
        return h, C

    def forward(self, ids, tf=1.0):
        B, T = ids.shape
        h, C = self.zeros(B)
        outs  = []
        for t in range(T-1):
            tok  = ids[:,t] if (t==0 or random.random()<tf) else outs[-1].argmax(-1)
            x    = self.emb(tok)
            h, C = self.cell(x, h, C)
            outs.append(self.out(h))
        return torch.stack(outs, dim=1)

# ============================================================
# VANILLA MODEL
# ============================================================
class VanillaModel(nn.Module):
    def __init__(self, vsz, hid=260):
        super().__init__()
        self.hid  = hid
        self.emb  = nn.Embedding(vsz, hid)
        self.cell = nn.RNNCell(hid, hid)
        self.out  = nn.Linear(hid, vsz)

    def forward(self, ids, tf=1.0):
        B, T = ids.shape
        h    = torch.zeros(B, self.hid, device=device)
        outs = []
        for t in range(T-1):
            tok  = ids[:,t] if (t==0 or random.random()<tf) else outs[-1].argmax(-1)
            x    = self.emb(tok)
            h    = self.cell(x, h)
            outs.append(self.out(h))
        return torch.stack(outs, dim=1)

# ============================================================
# LOSS
# ============================================================
def loss_and_acc(logits, ids, starts, pad_id):
    B, T, V = logits.shape
    targets = ids[:, 1:]
    mask    = (targets != pad_id)
    for b in range(B):
        if starts[b]-1 > 0:
            mask[b, :starts[b]-1] = False
    fl = logits.reshape(-1, V)
    ft = targets.reshape(-1)
    fm = mask.reshape(-1)
    if fm.sum() == 0:
        return torch.tensor(0.0, device=logits.device), 0.0
    loss = F.cross_entropy(fl[fm], ft[fm])
    acc  = (fl[fm].argmax(-1) == ft[fm]).float().mean().item()
    return loss, acc

# ============================================================
# GENERATION
# ============================================================
@torch.no_grad()
def gen_dppu(model, tok, prompt_ids, max_len=30):
    inp     = torch.tensor([prompt_ids], dtype=torch.long, device=device)
    h, C    = model.zeros(1)
    for t in range(inp.size(1)-1):
        x    = model.emb(inp[:,t])
        h, C = model.cell(x, h, C)
    cur, out = inp[:,-1], []
    for _ in range(max_len):
        x    = model.emb(cur)
        h, C = model.cell(x, h, C)
        lg   = model.out(h)
        nxt  = lg.argmax(-1)
        char = tok.inv.get(nxt.item(), '?')
        if char in ('<eos>','<pad>'): break
        out.append(char); cur = nxt
    return ''.join(out)

@torch.no_grad()
def gen_vanilla(model, tok, prompt_ids, max_len=30):
    inp = torch.tensor([prompt_ids], dtype=torch.long, device=device)
    h   = torch.zeros(1, model.hid, device=device)
    for t in range(inp.size(1)-1):
        x = model.emb(inp[:,t])
        h = model.cell(x, h)
    cur, out = inp[:,-1], []
    for _ in range(max_len):
        x  = model.emb(cur)
        h  = model.cell(x, h)
        lg = model.out(h)
        nxt  = lg.argmax(-1)
        char = tok.inv.get(nxt.item(), '?')
        if char in ('<eos>','<pad>'): break
        out.append(char); cur = nxt
    return ''.join(out)

def extract_answer(output):
    # pull the number after 'ans:' if present
    if 'ans:' in output:
        try: return output.split('ans:')[-1].split(',')[0].strip()
        except: pass
    return output

# ============================================================
# PROBE
# ============================================================
PROBE_EX = [
    (12, 34, 46),
    (55, 27, 82),
    (73, 18, 91),
    (99, 11, 110),
    (64, 36, 100),
]

@torch.no_grad()
def run_probe(dppu, vanilla, tok, step):
    dppu.eval(); vanilla.eval()
    d_ok = v_ok = 0
    lines = [f"\n  PROBE step={step}  anchor={dppu.cell.anchor.item():.6f}"]
    lines.append(f"  {'PROBLEM':<12} {'TARGET':<6} {'DPPU OUTPUT':<28} {'VANILLA OUTPUT':<28} {'ANS'}")
    lines.append(f"  {'-'*85}")
    for a, b, ans in PROBE_EX:
        prompt  = f"{a}+{b}="
        target  = make_step_example(a, b).split('=')[1]
        ids     = tok.encode(prompt, add_bos=True, add_eos=False)
        dp_out  = gen_dppu(dppu, tok, ids)
        vp_out  = gen_vanilla(vanilla, tok, ids)
        dp_ans  = extract_answer(dp_out)
        vp_ans  = extract_answer(vp_out)
        dh = '✓' if dp_ans == str(ans) else '✗'
        vh = '✓' if vp_ans == str(ans) else '✗'
        if dp_ans == str(ans): d_ok += 1
        if vp_ans == str(ans): v_ok += 1
        lines.append(f"  {prompt:<12} {str(ans):<6} {dh} {dp_out:<26} {vh} {vp_out:<26}")
    lines.append(f"\n  DPPU: {d_ok}/5 correct answers    VANILLA: {v_ok}/5 correct answers\n")
    msg = '\n'.join(lines)
    print(msg)
    with open(LOG_PATH,'a') as f: f.write(msg+'\n')
    sys.stdout.flush()
    dppu.train(); vanilla.train()

# ============================================================
# MAIN
# ============================================================
tok     = MathTokenizer()
dppu    = DPPUModel(tok.vocab_size, CFG['hidden']).to(device)
vanilla = VanillaModel(tok.vocab_size, CFG['hidden']).to(device)
d_opt   = torch.optim.Adam(dppu.parameters(),    lr=CFG['lr'])
v_opt   = torch.optim.Adam(vanilla.parameters(), lr=CFG['lr'])

d_params = sum(p.numel() for p in dppu.parameters())
v_params = sum(p.numel() for p in vanilla.parameters())

# print a sample to show the format
sample = gen_add2()
hdr = (f"\n{'='*62}\n"
       f"  DPPU-VRU v21  |  TEACH THE PROCESS\n"
       f"  device={device}\n"
       f"  DPPU params={d_params:,}  |  Vanilla params={v_params:,}\n"
       f"\n  Example training sequence:\n"
       f"  INPUT:  {sample.split('=')[0]}=\n"
       f"  TARGET: {sample.split('=')[1]}\n"
       f"\n  Model must predict: ones -> carry -> tens -> answer\n"
       f"  Not just the answer. The whole process.\n"
       f"{'='*62}\n")
print(hdr)
with open(LOG_PATH,'a') as f: f.write(hdr)

tf = CFG['tf_start']
dl = da = vl = va = rc = 0.0

for step in range(1, CFG['max_steps']+1):
    tf = max(CFG['tf_min'], tf * CFG['tf_decay'])
    ids, starts = make_batch(CFG['batch'], tok)

    # DPPU
    dppu.train()
    d_opt.zero_grad()
    d_logits = dppu(ids, tf=tf)
    d_loss, d_acc = loss_and_acc(d_logits, ids, starts, tok.pad_id)
    d_loss.backward()
    torch.nn.utils.clip_grad_norm_(dppu.parameters(), 5.0)
    d_opt.step()

    # Vanilla
    vanilla.train()
    v_opt.zero_grad()
    v_logits = vanilla(ids, tf=tf)
    v_loss, v_acc = loss_and_acc(v_logits, ids, starts, tok.pad_id)
    v_loss.backward()
    torch.nn.utils.clip_grad_norm_(vanilla.parameters(), 5.0)
    v_opt.step()

    dl += d_loss.item(); da += d_acc
    vl += v_loss.item(); va += v_acc
    rc += 1

    if step % CFG['log_every'] == 0:
        msg = (f"step={step:6d} | "
               f"DPPU  loss={dl/rc:.4f} acc={da/rc*100:.1f}%  anchor={dppu.cell.anchor.item():.4f} | "
               f"VANILLA  loss={vl/rc:.4f} acc={va/rc*100:.1f}% | "
               f"tf={tf:.3f}")
        print(msg)
        with open(LOG_PATH,'a') as f: f.write(msg+'\n')
        sys.stdout.flush()
        dl = da = vl = va = rc = 0.0

    if step % CFG['probe_every'] == 0:
        run_probe(dppu, vanilla, tok, step)

print(f"\nDONE -- step={step}")
print(f"Final anchor: {dppu.cell.anchor.item():.6f}  (started at {PHI_CL:.6f})")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

  DPPU-VRU v21  |  TEACH THE PROCESS
  device=cuda
  DPPU params=229,890  |  Vanilla params=161,249

  Example training sequence:
  INPUT:  84+28=
  TARGET: ones:12,carry:1,tens:11,ans:112

  Model must predict: ones -> carry -> tens -> answer
  Not just the answer. The whole process.



KeyboardInterrupt: 

In [ ]:
# ============================================================
# DPPU-VRU v21 -- TEACH THE PROCESS NOT THE ANSWER
# Dylan Michael Scott -- Horizon Tech
#
# Instead of: 73+18=91
# We give it:  73+18=ones:11,carry:1,tens:9,ans:91
#
# The model follows the actual computation steps.
# Ones column first. Carry. Tens column. Then answer.
# This mirrors how arithmetic is actually performed --
# not as a lookup, but as a sequential process.
#
# DPPU vs Vanilla side by side. Same batches. Honest test.
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import torch, torch.nn as nn, torch.nn.functional as F
import math, random, os, json, sys
from datetime import datetime

DRIVE_DIR = '/content/drive/MyDrive/dppu_vru'
LOG_PATH  = os.path.join(DRIVE_DIR, 'v21_log.txt')
os.makedirs(DRIVE_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

PI_CL  = math.pi
PHI_CL = 4.0 / math.pi
D_CAP  = 4
EPS    = 1e-7

def pi_dyn(d):  return 4.0 - (4.0 - PI_CL) * torch.exp(-d)
def phi_dyn(d): return PHI_CL * torch.exp(-d) + (1.0 - torch.exp(-d))
def omega(d):   return (pi_dyn(d) * phi_dyn(d)) / (1.0 + d + EPS)

def compute_delta(h):
    return torch.tanh(torch.abs(h) / (h.std(dim=-1, keepdim=True) + EPS))

CFG = dict(
    hidden       = 260,
    lr           = 1e-3,
    batch        = 32,
    max_steps    = 20_000,
    log_every    = 200,
    probe_every  = 2_000,
    tf_start     = 0.9,
    tf_min       = 0.05,
    tf_decay     = 0.9998,  # slower -- model solidifies before free-gen
)

# ============================================================
# TOKENIZER -- extended for step tokens
# ============================================================
class MathTokenizer:
    SPECIAL = ['<pad>', '<bos>', '<eos>', '<unk>']

    def __init__(self):
        self.vocab = {s: i for i, s in enumerate(self.SPECIAL)}
        # digits, operators, separators
        for c in '0123456789+-*/=:,abcdefghijklmnopqrstuvwxyz_ ':
            if c not in self.vocab:
                self.vocab[c] = len(self.vocab)
        self.inv = {v: k for k, v in self.vocab.items()}
        self.pad_id = self.vocab['<pad>']
        self.bos_id = self.vocab['<bos>']
        self.eos_id = self.vocab['<eos>']
        self.eq_id  = self.vocab['=']

    @property
    def vocab_size(self): return len(self.vocab)

    def encode(self, text, add_bos=True, add_eos=True):
        return (([self.bos_id] if add_bos else []) +
                [self.vocab.get(c, self.vocab['<unk>']) for c in text] +
                ([self.eos_id] if add_eos else []))

    def decode(self, ids):
        return ''.join(self.inv.get(i,'?') for i in ids
                       if self.inv.get(i,'?') not in self.SPECIAL)

# ============================================================
# DATA -- step-by-step format
#
# 73+18= -> ones:11,carry:1,tens:9,ans:91
#
# The answer is still at the end (ans:XX)
# but the model has to work through the process to get there
# ============================================================
def make_step_example(a, b):
    ones      = (a % 10) + (b % 10)
    carry     = ones // 10
    tens_sum  = (a // 10) + (b // 10) + carry
    answer    = a + b
    # format: question=ones:N,carry:N,tens:N,ans:N
    prompt  = f"{a}+{b}="
    process = f"ones:{ones},carry:{carry},tens:{tens_sum},ans:{answer}"
    return prompt + process

def gen_add2():
    a, b = random.randint(10,99), random.randint(10,99)
    return make_step_example(a, b)

def make_batch(bsz, tok):
    seqs, starts = [], []
    for _ in range(bsz):
        raw   = gen_add2()
        ids   = tok.encode(raw)
        # answer starts after '=' -- model predicts the process
        start = next((j+1 for j,t in enumerate(ids) if t==tok.eq_id), 1)
        seqs.append(ids); starts.append(start)
    ml  = max(len(s) for s in seqs)
    pad = [s + [tok.pad_id]*(ml-len(s)) for s in seqs]
    return torch.tensor(pad, dtype=torch.long, device=device), starts

# ============================================================
# DPPU MODEL
# ============================================================
class DPPUCell(nn.Module):
    def __init__(self, in_dim, hid):
        super().__init__()
        self.n   = D_CAP + 1
        self.ds  = hid // self.n
        self.hid = hid
        self.Wx  = nn.Linear(in_dim, hid)
        self.Wh  = nn.Linear(hid, hid, bias=False)
        self.Wc  = nn.Linear(hid, self.n, bias=False)
        self.Wo  = nn.Linear(hid, hid, bias=False)
        self.anchor = nn.Parameter(torch.tensor(PHI_CL))

    def forward(self, x, h, C):
        B   = x.size(0)
        xp  = self.Wx(x)
        wh  = self.Wh(h)
        h_  = h.view(B, self.n, self.ds)
        xp_ = xp.view(B, self.n, self.ds)
        wh_ = wh.view(B, self.n, self.ds)
        di     = torch.tanh(torch.abs(h_) / (h_.std(dim=-1, keepdim=True) + EPS))
        h_rec  = (phi_dyn(di) / PHI_CL) * wh_
        x_rec  = (pi_dyn(di)  / PI_CL)  * xp_
        gate   = torch.sigmoid(omega(di))
        anchor = C.unsqueeze(-1) * self.anchor
        h_new  = torch.tanh(h_rec + x_rec + anchor) * gate
        h2 = self.Wo(h_new.view(B, self.hid))
        C2 = torch.clamp(torch.tanh(0.1*C + 0.01*self.Wc(h2)), -0.5, 0.5)
        return h2, C2


class DPPUModel(nn.Module):
    def __init__(self, vsz, hid=260):
        super().__init__()
        self.hid  = hid
        self.n    = D_CAP + 1
        self.emb  = nn.Embedding(vsz, hid)
        self.cell = DPPUCell(hid, hid)
        self.out  = nn.Linear(hid, vsz)

    def zeros(self, B):
        h = torch.zeros(B, self.hid, device=device)
        C = torch.full((B, self.n), PHI_CL, device=device)
        return h, C

    def forward(self, ids, tf=1.0):
        B, T = ids.shape
        h, C = self.zeros(B)
        outs  = []
        for t in range(T-1):
            tok  = ids[:,t] if (t==0 or random.random()<tf) else outs[-1].argmax(-1)
            x    = self.emb(tok)
            h, C = self.cell(x, h, C)
            outs.append(self.out(h))
        return torch.stack(outs, dim=1)

# ============================================================
# VANILLA MODEL
# ============================================================
class VanillaModel(nn.Module):
    def __init__(self, vsz, hid=260):
        super().__init__()
        self.hid  = hid
        self.emb  = nn.Embedding(vsz, hid)
        self.cell = nn.RNNCell(hid, hid)
        self.out  = nn.Linear(hid, vsz)

    def forward(self, ids, tf=1.0):
        B, T = ids.shape
        h    = torch.zeros(B, self.hid, device=device)
        outs = []
        for t in range(T-1):
            tok  = ids[:,t] if (t==0 or random.random()<tf) else outs[-1].argmax(-1)
            x    = self.emb(tok)
            h    = self.cell(x, h)
            outs.append(self.out(h))
        return torch.stack(outs, dim=1)

# ============================================================
# LOSS
# ============================================================
def loss_and_acc(logits, ids, starts, pad_id):
    B, T, V = logits.shape
    targets = ids[:, 1:]
    mask    = (targets != pad_id)
    for b in range(B):
        if starts[b]-1 > 0:
            mask[b, :starts[b]-1] = False
    fl = logits.reshape(-1, V)
    ft = targets.reshape(-1)
    fm = mask.reshape(-1)
    if fm.sum() == 0:
        return torch.tensor(0.0, device=logits.device), 0.0
    loss = F.cross_entropy(fl[fm], ft[fm])
    acc  = (fl[fm].argmax(-1) == ft[fm]).float().mean().item()
    return loss, acc

# ============================================================
# GENERATION
# ============================================================
@torch.no_grad()
def gen_dppu(model, tok, prompt_ids, max_len=30):
    inp     = torch.tensor([prompt_ids], dtype=torch.long, device=device)
    h, C    = model.zeros(1)
    for t in range(inp.size(1)-1):
        x    = model.emb(inp[:,t])
        h, C = model.cell(x, h, C)
    cur, out = inp[:,-1], []
    for _ in range(max_len):
        x    = model.emb(cur)
        h, C = model.cell(x, h, C)
        lg   = model.out(h)
        nxt  = lg.argmax(-1)
        char = tok.inv.get(nxt.item(), '?')
        if char in ('<eos>','<pad>'): break
        out.append(char); cur = nxt
    return ''.join(out)

@torch.no_grad()
def gen_vanilla(model, tok, prompt_ids, max_len=30):
    inp = torch.tensor([prompt_ids], dtype=torch.long, device=device)
    h   = torch.zeros(1, model.hid, device=device)
    for t in range(inp.size(1)-1):
        x = model.emb(inp[:,t])
        h = model.cell(x, h)
    cur, out = inp[:,-1], []
    for _ in range(max_len):
        x  = model.emb(cur)
        h  = model.cell(x, h)
        lg = model.out(h)
        nxt  = lg.argmax(-1)
        char = tok.inv.get(nxt.item(), '?')
        if char in ('<eos>','<pad>'): break
        out.append(char); cur = nxt
    return ''.join(out)

def extract_answer(output):
    # pull the number after 'ans:' if present
    if 'ans:' in output:
        try: return output.split('ans:')[-1].split(',')[0].strip()
        except: pass
    return output

# ============================================================
# PROBE
# ============================================================
PROBE_EX = [
    (12, 34, 46),
    (55, 27, 82),
    (73, 18, 91),
    (99, 11, 110),
    (64, 36, 100),
]

@torch.no_grad()
def run_probe(dppu, vanilla, tok, step):
    dppu.eval(); vanilla.eval()
    d_ok = v_ok = 0
    lines = [f"\n  PROBE step={step}  anchor={dppu.cell.anchor.item():.6f}"]
    lines.append(f"  {'PROBLEM':<12} {'TARGET':<6} {'DPPU OUTPUT':<28} {'VANILLA OUTPUT':<28} {'ANS'}")
    lines.append(f"  {'-'*85}")
    for a, b, ans in PROBE_EX:
        prompt  = f"{a}+{b}="
        target  = make_step_example(a, b).split('=')[1]
        ids     = tok.encode(prompt, add_bos=True, add_eos=False)
        dp_out  = gen_dppu(dppu, tok, ids)
        vp_out  = gen_vanilla(vanilla, tok, ids)
        dp_ans  = extract_answer(dp_out)
        vp_ans  = extract_answer(vp_out)
        dh = '✓' if dp_ans == str(ans) else '✗'
        vh = '✓' if vp_ans == str(ans) else '✗'
        if dp_ans == str(ans): d_ok += 1
        if vp_ans == str(ans): v_ok += 1
        lines.append(f"  {prompt:<12} {str(ans):<6} {dh} {dp_out:<26} {vh} {vp_out:<26}")
    lines.append(f"\n  DPPU: {d_ok}/5 correct answers    VANILLA: {v_ok}/5 correct answers\n")
    msg = '\n'.join(lines)
    print(msg)
    with open(LOG_PATH,'a') as f: f.write(msg+'\n')
    sys.stdout.flush()
    dppu.train(); vanilla.train()

# ============================================================
# MAIN
# ============================================================
tok     = MathTokenizer()
dppu    = DPPUModel(tok.vocab_size, CFG['hidden']).to(device)
vanilla = VanillaModel(tok.vocab_size, CFG['hidden']).to(device)
d_opt   = torch.optim.Adam(dppu.parameters(),    lr=CFG['lr'])
v_opt   = torch.optim.Adam(vanilla.parameters(), lr=CFG['lr'])

d_params = sum(p.numel() for p in dppu.parameters())
v_params = sum(p.numel() for p in vanilla.parameters())

# print a sample to show the format
sample = gen_add2()
hdr = (f"\n{'='*62}\n"
       f"  DPPU-VRU v21  |  TEACH THE PROCESS\n"
       f"  device={device}\n"
       f"  DPPU params={d_params:,}  |  Vanilla params={v_params:,}\n"
       f"\n  Example training sequence:\n"
       f"  INPUT:  {sample.split('=')[0]}=\n"
       f"  TARGET: {sample.split('=')[1]}\n"
       f"\n  Model must predict: ones -> carry -> tens -> answer\n"
       f"  Not just the answer. The whole process.\n"
       f"{'='*62}\n")
print(hdr)
with open(LOG_PATH,'a') as f: f.write(hdr)

tf = CFG['tf_start']
dl = da = vl = va = rc = 0.0

for step in range(1, CFG['max_steps']+1):
    tf = max(CFG['tf_min'], tf * CFG['tf_decay'])
    ids, starts = make_batch(CFG['batch'], tok)

    # DPPU
    dppu.train()
    d_opt.zero_grad()
    d_logits = dppu(ids, tf=tf)
    d_loss, d_acc = loss_and_acc(d_logits, ids, starts, tok.pad_id)
    d_loss.backward()
    torch.nn.utils.clip_grad_norm_(dppu.parameters(), 5.0)
    d_opt.step()

    # Vanilla
    vanilla.train()
    v_opt.zero_grad()
    v_logits = vanilla(ids, tf=tf)
    v_loss, v_acc = loss_and_acc(v_logits, ids, starts, tok.pad_id)
    v_loss.backward()
    torch.nn.utils.clip_grad_norm_(vanilla.parameters(), 5.0)
    v_opt.step()

    dl += d_loss.item(); da += d_acc
    vl += v_loss.item(); va += v_acc
    rc += 1

    if step % CFG['log_every'] == 0:
        msg = (f"step={step:6d} | "
               f"DPPU  loss={dl/rc:.4f} acc={da/rc*100:.1f}%  anchor={dppu.cell.anchor.item():.4f} | "
               f"VANILLA  loss={vl/rc:.4f} acc={va/rc*100:.1f}% | "
               f"tf={tf:.3f}")
        print(msg)
        with open(LOG_PATH,'a') as f: f.write(msg+'\n')
        sys.stdout.flush()
        dl = da = vl = va = rc = 0.0

    if step % CFG['probe_every'] == 0:
        run_probe(dppu, vanilla, tok, step)

print(f"\nDONE -- step={step}")
print(f"Final anchor: {dppu.cell.anchor.item():.6f}  (started at {PHI_CL:.6f})")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

  DPPU-VRU v21  |  TEACH THE PROCESS
  device=cuda
  DPPU params=229,890  |  Vanilla params=161,249

  Example training sequence:
  INPUT:  54+34=
  TARGET: ones:8,carry:0,tens:8,ans:88

  Model must predict: ones -> carry -> tens -> answer
  Not just the answer. The whole process.

step=   200 | DPPU  loss=0.5941 acc=82.4%  anchor=1.2478 | VANILLA  loss=0.5939 acc=83.0% | tf=0.865
step=   400 | DPPU  loss=0.4071 acc=86.0%  anchor=1.2320 | VANILLA  loss=0.3711 acc=87.4% | tf=0.831
step=   600 | DPPU  loss=0.3561 acc=87.8%  anchor=1.2344 | VANILLA  loss=0.3549 acc=87.7% | tf=0.798
step=   800 | DPPU  loss=0.3847 acc=87.3%  anchor=1.2285 | VANILLA  loss=0.3350 acc=88.5% | tf=0.767
step=  1000 | DPPU  loss=0.3758 acc=87.6%  anchor=1.2263 | VANILLA  loss=0.3151 acc=89.1% | tf=0.737
step=  1200 | DPPU  loss=0.3789 acc=87.1%  anchor=1.2148 | VANILLA  loss=0.3073 a